In [ ]:
# ══════════════════════════════════════════════
# PRELUDE — run this cell first
# Added by fix pass. Everything below your own code is unchanged.
# ══════════════════════════════════════════════

import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from datetime import datetime, timezone

class Stale(RuntimeError): pass
class Unconverged(RuntimeError): pass
class TooFewObs(RuntimeError): pass


def safe_at(obj, i=-1, col=None):
    """float() on a 1-element Series is deprecated. Handles yfinance MultiIndex columns."""
    s = obj[col] if col is not None else obj
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    a = np.asarray(s.dropna()).ravel()
    if a.size == 0:
        raise Stale("empty series")
    return float(a[i])


def safe_last(obj, col=None):
    return safe_at(obj, -1, col)


def unmute_convergence():
    """Let convergence failures through. They were being swallowed by filterwarnings('ignore')."""
    try:
        import statsmodels.tools.sm_exceptions as _s
        for _n in ("ConvergenceWarning", "EstimationWarning", "ValueWarning"):
            if hasattr(_s, _n):
                warnings.filterwarnings("always", category=getattr(_s, _n))
    except Exception:
        pass
    try:
        from arch.utility.exceptions import DataScaleWarning
        warnings.filterwarnings("always", category=DataScaleWarning)
    except Exception:
        pass


# ── CONTRACT: pin which gold series this notebook uses ──────────────
#   'front_intraday' = GC=F 30-min  (the contract you trade — DEFAULT)
#   'front_daily'    = GC=F daily   (often prints spot, not the future)
#   'spot'           = XAUUSD
LIVE_CONTRACT = "front_intraday"


def get_ctx(contract=LIVE_CONTRACT):
    if contract == "front_intraday":
        gc_df = yf.download("GC=F", period="5d", interval="30m", progress=False)
    elif contract == "front_daily":
        gc_df = yf.download("GC=F", period="1mo", interval="1d", progress=False)
    elif contract == "spot":
        gc_df = yf.download("XAUUSD=X", period="5d", interval="30m", progress=False)
    else:
        raise ValueError(contract)

    gc = safe_last(gc_df, "Close")
    gld = safe_last(yf.download("GLD", period="5d", progress=False), "Close")

    def _s(t, d):
        try:
            return safe_last(yf.download(t, period="5d", progress=False), "Close")
        except Exception:
            return d

    if not 1000 < gc < 12000:
        raise Stale(f"GC={gc:,.2f} implausible — bad fetch")
    if not 100 < gld < 1200:
        raise Stale(f"GLD={gld:,.2f} implausible — bad fetch")
    ratio = gc / gld
    if not 9.5 <= ratio <= 12.5:
        raise Stale(f"GLD->gold ratio {ratio:.3f}x outside [9.5, 12.5] — "
                    f"prices are from different dates")

    # known failure point: GC=F daily and 30m disagree
    try:
        _d = safe_last(yf.download("GC=F", period="1mo", interval="1d", progress=False), "Close")
        _i = safe_last(yf.download("GC=F", period="5d", interval="30m", progress=False), "Close")
        if abs(_d - _i) / _i > 0.005:
            print(f"  !! GC=F daily {_d:,.2f} vs 30m {_i:,.2f} — ${abs(_d-_i):,.1f} apart.")
            print(f"     Using '{contract}'. VERIFY THE SETTLE IN QUANTOWER.")
    except Exception:
        pass

    ctx = dict(gc=gc, gld=gld, ratio=ratio, contract=contract,
               asof=str(gc_df.index[-1]),
               vix=_s("^VIX", np.nan), gvz=_s("^GVZ", np.nan), rf=_s("^IRX", 4.0) / 100)
    print(f"  contract : {contract}")
    print(f"  GC {gc:>10,.2f}   GLD {gld:>8,.2f}   ratio {ratio:.4f}x   <-- NOT 10.0")
    print(f"  VIX {ctx['vix']:.2f}   GVZ {ctx['gvz']:.1f}   rf {ctx['rf']:.2%}   as of {ctx['asof']}")
    return ctx


def need_obs(n, floor, label=""):
    if n < floor:
        raise TooFewObs(f"{label}: {n} observations, need >= {floor}. Do not report this fit.")


def need_fresh(as_of, days=7, label="field"):
    age = (datetime.now(timezone.utc).date() - datetime.fromisoformat(as_of).date()).days
    if age > days:
        raise Stale(f"{label} written {as_of} ({age}d ago) — EXPIRED. Rewrite or delete it.")
    return True


def need_converged(res, states=None, label="model"):
    conv = getattr(res, "converged", None)
    if conv is None and isinstance(getattr(res, "mle_retvals", None), dict):
        conv = res.mle_retvals.get("converged", True)
    if conv is False:
        raise Unconverged(f"{label}: optimiser did not converge. Not a regime classification.")
    if states is not None:
        u, c = np.unique(np.asarray(states), return_counts=True)
        if len(u) < 2:
            raise Unconverged(f"{label}: {len(u)} distinct state — a flat line, not regimes.")
        if c.min() / c.sum() < 0.05:
            raise Unconverged(f"{label}: minority state is {c.min()/c.sum():.1%} — "
                              f"outlier detector, not a regime model.")


def kelly_cap(f, frac=0.25, cap=0.20):
    out = float(np.clip(f * frac, -cap, cap))
    if abs(f) > 1:
        print(f"  ! raw f*={f:.2f} implies {f*100:.0f}% of capital (small-sample artefact). "
              f"Using {out:.1%}.")
    return out


LIVE_CTX   = get_ctx()
LIVE_RATIO = LIVE_CTX["ratio"]   # use instead of 10
LIVE_GC    = LIVE_CTX["gc"]

# NOTE: named LIVE_* on purpose — GOLD is your hex colour in 15 cells,
#       and RATIO / CONTRACT are already used in cells 44-45.


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# COT AUTO-FETCH — CFTC Disaggregated, Futures-and-Options COMBINED
# Source: https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm
#
# NOTE: your old code pulled f_year.txt from fut_disagg_* = FUTURES ONLY,
#       while every dashboard header said "Options and Futures Combined".
#       This uses com_disagg_* -> c_year.txt = actually COMBINED.
#
# The annual zip is rewritten by CFTC every Friday ~15:30 ET, so pulling
# the current year's file always gives the newest report. No manual step.
# ══════════════════════════════════════════════════════════════════════

import io, os, zipfile, requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path

COT_CACHE = Path.home() / ".cot_cache"
COT_CACHE.mkdir(exist_ok=True)

# Disaggregated Futures-and-Options Combined, annual history
COT_HIST_URL = "https://www.cftc.gov/files/dea/history/com_disagg_txt_{year}.zip"
# Current-week snapshot (headerless; used only as a freshness cross-check)
COT_CURRENT_URL = "https://www.cftc.gov/dea/newcot/c_disagg.txt"

CFTC_CODES = {
    "gold":        "088691",
    "silver":      "084691",
    "copper":      "085692",
    "wti_crude":   "067651",
    "natgas":      "023651",
    "corn":        "002602",
    "platinum":    "076651",
    "palladium":   "075651",
}

_COT_UA = {"User-Agent": "Mozilla/5.0 (research; contact: elena)"}


def _cot_expected_report_date(now=None):
    """
    COT is Tuesday data released Friday 15:30 ET. Returns the latest
    Tuesday that should be published by now.
    """
    now = now or datetime.now(timezone.utc)
    et = now - timedelta(hours=4)                    # ET approx (EDT)
    days_since_fri = (et.weekday() - 4) % 7
    last_fri = (et - timedelta(days=days_since_fri)).replace(
        hour=15, minute=30, second=0, microsecond=0)
    if et < last_fri:
        last_fri -= timedelta(days=7)
    return (last_fri - timedelta(days=3)).date()     # the Tuesday it covers


def _cot_download_year(year, force=False):
    """Download and cache one year of combined disaggregated data."""
    cache = COT_CACHE / f"com_disagg_{year}.parquet"
    stamp = COT_CACHE / f"com_disagg_{year}.stamp"

    # current year: refresh if cache older than 12h; past years never change
    fresh = False
    if cache.exists() and not force:
        if year < datetime.now().year:
            fresh = True
        elif stamp.exists():
            age_h = (datetime.now().timestamp() - float(stamp.read_text())) / 3600
            fresh = age_h < 12
    if fresh:
        return pd.read_parquet(cache)

    url = COT_HIST_URL.format(year=year)
    r = requests.get(url, timeout=60, headers=_COT_UA)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        names = z.namelist()
        inner = next((n for n in names if n.lower().endswith((".txt", ".csv"))), None)
        if inner is None:
            raise RuntimeError(f"{year}: no txt/csv in zip — got {names}")
        if inner.lower().startswith("f_"):
            raise RuntimeError(
                f"{year}: zip contains '{inner}' (f_ = FUTURES ONLY). "
                f"Expected 'c_year.txt' from com_disagg_*. Wrong URL.")
        with z.open(inner) as fh:
            df = pd.read_csv(fh, low_memory=False)

    df.columns = [c.strip() for c in df.columns]
    cache.parent.mkdir(exist_ok=True)
    df.to_parquet(cache, index=False)
    stamp.write_text(str(datetime.now().timestamp()))
    print(f"    downloaded {year}  ({inner}, {len(df):,} rows)")
    return df


def _cot_date_col(df):
    for c in ("Report_Date_as_YYYY-MM-DD", "Report_Date_as_MM_DD_YYYY",
              "As_of_Date_In_Form_YYMMDD", "Report_Date"):
        if c in df.columns:
            return c
    raise KeyError(f"no date column found in {list(df.columns)[:12]}")


def _cot_parse_dates(s, col):
    if "YYMMDD" in col:
        return pd.to_datetime(s.astype(str).str.zfill(6), format="%y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")


def fetch_cot(market="gold", years=3, force=False, verbose=True):
    """
    Returns a tidy weekly DataFrame for one market, newest last:

        date, open_interest,
        prod_long, prod_short, prod_net,
        swap_long, swap_short, swap_net,
        mm_long,   mm_short,   mm_net,
        other_net, comm_net (prod+swap),
        mm_index, comm_index   (156-week rolling 0-100 Williams index)

    Raises on anything implausible rather than returning zeros.
    """
    code = CFTC_CODES.get(market.lower(), market)
    this_year = datetime.now().year
    frames = []
    if verbose:
        print(f"  CFTC Disaggregated · Futures-and-Options COMBINED · code {code}")

    for y in range(this_year - years + 1, this_year + 1):
        try:
            frames.append(_cot_download_year(y, force=force))
        except Exception as e:
            print(f"    ! {y}: {type(e).__name__}: {e}")
    if not frames:
        raise RuntimeError("no COT data retrieved — check network / CFTC availability")

    raw = pd.concat(frames, ignore_index=True)

    code_col = next((c for c in ("CFTC_Contract_Market_Code", "CFTC_Contract_Market_Code_Quotes")
                     if c in raw.columns), None)
    if code_col is None:
        raise KeyError("no CFTC_Contract_Market_Code column")

    m = raw[raw[code_col].astype(str).str.strip().str.zfill(6) == code].copy()
    if m.empty:
        names = raw["Market_and_Exchange_Names"].dropna().unique()[:5]
        raise ValueError(f"code {code} not found. Sample markets: {list(names)}")

    dcol = _cot_date_col(m)
    m["date"] = _cot_parse_dates(m[dcol], dcol)
    m = m.dropna(subset=["date"]).sort_values("date")
    m = m.drop_duplicates(subset=["date"], keep="last")

    def col(*cands):
        for c in cands:
            if c in m.columns:
                return pd.to_numeric(m[c], errors="coerce")
        raise KeyError(f"none of {cands} present")

    out = pd.DataFrame({
        "date":          m["date"].values,
        "open_interest": col("Open_Interest_All"),
        "prod_long":     col("Prod_Merc_Positions_Long_All"),
        "prod_short":    col("Prod_Merc_Positions_Short_All"),
        "swap_long":     col("Swap_Positions_Long_All"),
        "swap_short":    col("Swap__Positions_Short_All", "Swap_Positions_Short_All"),
        "mm_long":       col("M_Money_Positions_Long_All"),
        "mm_short":      col("M_Money_Positions_Short_All"),
        "other_long":    col("Other_Rept_Positions_Long_All"),
        "other_short":   col("Other_Rept_Positions_Short_All"),
    }).reset_index(drop=True)

    out["prod_net"]  = out.prod_long  - out.prod_short
    out["swap_net"]  = out.swap_long  - out.swap_short
    out["mm_net"]    = out.mm_long    - out.mm_short
    out["other_net"] = out.other_long - out.other_short
    out["comm_net"]  = out.prod_net + out.swap_net     # producers + swap dealers

    w = min(156, len(out))
    for c in ("mm_net", "comm_net", "swap_net"):
        lo = out[c].rolling(w, min_periods=20).min()
        hi = out[c].rolling(w, min_periods=20).max()
        out[c.replace("_net", "_index")] = 100 * (out[c] - lo) / (hi - lo).replace(0, np.nan)

    # ---- validation: fail loud rather than score a broken parse ----------
    last = out.iloc[-1]
    if last.open_interest < 10_000:
        raise ValueError(f"open interest {last.open_interest:,.0f} implausible — bad parse")
    for f in ("prod_net", "swap_net", "mm_net"):
        if last[f] == 0:
            raise ValueError(f"{f} parsed as exactly 0 — column mismatch, not flat positioning")

    expected = _cot_expected_report_date()
    lag = (expected - last.date.date()).days
    if lag > 7:
        print(f"    !! newest report {last.date.date()} but {expected} expected "
              f"({lag}d stale). CFTC may be delayed, or cache is stale — force=True to refresh.")
    elif verbose:
        print(f"    latest report : {last.date.date()}  (current)")

    if verbose:
        print(f"    rows          : {len(out)}  [{out.date.min().date()} -> {out.date.max().date()}]")
        print(f"    OI            : {last.open_interest:>10,.0f}")
        print(f"    Managed Money : {last.mm_net:>+10,.0f}   index {last.mm_index:5.1f}/100")
        print(f"    Swap Dealers  : {last.swap_net:>+10,.0f}   index {last.swap_index:5.1f}/100")
        print(f"    Prod/Merch    : {last.prod_net:>+10,.0f}")
        print(f"    Commercial    : {last.comm_net:>+10,.0f}   index {last.comm_index:5.1f}/100")

    return out


def cot_signal(df, extreme_lo=20, extreme_hi=80):
    """Williams rule: need BOTH commercial and managed-money at an extreme."""
    r = df.iloc[-1]
    c, m = r.comm_index, r.mm_index
    if np.isnan(c) or np.isnan(m):
        return "INSUFFICIENT HISTORY", 0
    if c >= extreme_hi and m <= extreme_lo:
        return "BULLISH — commercials long, specs washed out", +1
    if c <= extreme_lo and m >= extreme_hi:
        return "BEARISH — commercials short, specs crowded", -1
    return f"NEUTRAL — no extreme (comm {c:.0f}, mm {m:.0f})", 0


Gold Weekly Prep Dash - Elena H.
¶
Sunday pre-week analysis across 4 tiers: Positioning · Macro · Options · Technicals

Framework: Larry Williams + institutional order flow

Gold is in a valid bearish setup this week (score -5). The smart money (commercials) are at their most short in 3 years, ETF money is flowing out, gold is below both its 200-day moving average and its 6-month average price, and the macro backdrop is hostile with oil-driven inflation pushing rate hike fears higher. There's no bullish signal firing anywhere. The play is to sell rallies into the 4,680–4,720 zone with a stop above 4,800 and a target of 4,493–4,540, and if that floor breaks, the next magnet is 4,300. Don't trade Monday (Memorial Day, thin and gappy), wait for Wednesday GDP and especially Thursday PCE inflation data before adding size, and keep one eye on Iran — a peace deal flushes gold to 4,300–4,400 fast, an escalation spikes it to 4,750+ and you cover immediately. The only thing that cancels the whole bearish thesis is a daily close above 4,894.

In [ ]:
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

import pandas as pd
import numpy as np
import yfinance as yf
import requests
import io, zipfile, re
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

plt.style.use('dark_background')
GOLD_COLOR    = '#FFD700'
BEAR_COLOR    = '#FF4444'
BULL_COLOR    = '#00FF88'
NEUTRAL_COLOR = '#888888'
BG_COLOR      = '#0a0a0a'
GRID_COLOR    = '#1a1a1a'

TODAY     = datetime.today()
START_3Y  = (TODAY - timedelta(days=3*365)).strftime('%Y-%m-%d')
START_1Y  = (TODAY - timedelta(days=365)).strftime('%Y-%m-%d')
START_6M  = (TODAY - timedelta(days=180)).strftime('%Y-%m-%d')
START_2Y  = (TODAY - timedelta(days=2*365)).strftime('%Y-%m-%d')

print(f" Weekly prep run: {TODAY.strftime('%A %d %B %Y')}")
print("=" * 55)


Tier 1 · Positioning — Who is where and how extreme
¶
Sources: CFTC COT (combined futures+options), GLD options OI, Large Spec Index

In [ ]:
# ── COT: Fetch live from CFTC CMX combined long-format report ──────────────
# Source: https://www.cftc.gov/dea/options/deacmxlof.htm  |  Gold code: 088691
COT_URL   = 'https://www.cftc.gov/dea/options/deacmxlof.htm'
GOLD_CODE = '088691'

def parse_cot_page(url, commodity_code):
    """Parse CFTC fixed-width long-format CMX page. Returns latest week's gold figures."""
    resp = requests.get(url, timeout=20, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    text = resp.text

    pattern = rf'GOLD.*?Code-{commodity_code}(.*?)(?=\n [A-Z].*?Code-\d{{6}}|\Z)'
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    if not match:
        raise ValueError(f"Gold block (code {commodity_code}) not found")
    block = match.group(0)

    date_match = re.search(r'(\w+ \d+, \d{4})', block)
    report_date = datetime.strptime(date_match.group(1), '%B %d, %Y') if date_match else None

    all_match = re.search(
        r'^All\s*:\s*([\d,]+):\s*([\d,]+)\s+([\d,]+)\s+([\d,]+)\s+([\d,]+)\s+([\d,]+)',
        block, re.MULTILINE)
    if not all_match:
        raise ValueError("Could not parse 'All' data row")

    def n(s): return int(s.replace(',', ''))
    oi        = n(all_match.group(1))
    nc_long   = n(all_match.group(2))
    nc_short  = n(all_match.group(3))
    nc_spread = n(all_match.group(4))
    comm_long = n(all_match.group(5))
    comm_short= n(all_match.group(6))

    chg_match = re.search(
        r':\s*(-?[\d,]+):\s*(-?[\d,]+)\s+(-?[\d,]+)\s+(-?[\d,]+)\s+(-?[\d,]+)\s+(-?[\d,]+)',
        block)
    chg = {}
    if chg_match:
        def nc(s): return int(s.replace(' ','').replace(',',''))
        chg['oi']       = nc(chg_match.group(1))
        chg['nc_long']  = nc(chg_match.group(2))
        chg['nc_short'] = nc(chg_match.group(3))
        chg['comm_long']= nc(chg_match.group(5))
        chg['comm_short']= nc(chg_match.group(6))
        chg['comm_net'] = chg['comm_long'] - chg['comm_short']
        chg['spec_net'] = chg['nc_long']   - chg['nc_short']

    return {
        'report_date' : report_date,
        'open_interest': oi,
        'comm_long'   : comm_long,
        'comm_short'  : comm_short,
        'comm_net'    : comm_long - comm_short,
        'spec_long'   : nc_long,
        'spec_short'  : nc_short,
        'spec_net'    : nc_long - nc_short,
        'spec_spread' : nc_spread,
        'changes'     : chg,
    }

try:
    cot_latest = parse_cot_page(COT_URL, GOLD_CODE)
    c  = cot_latest
    ch = c['changes']
    print("━" * 62)
    print(" CFTC COT — GOLD (COMEX, Combined F+O) Code-088691")
    print(f" Report date  : {c['report_date'].strftime('%d %B %Y') if c['report_date'] else 'unknown'}")
    print("━" * 62)
    print(f"  Open Interest      : {c['open_interest']:>10,}  Δ {ch.get('oi', 0):>+8,}")
    print(f"  Commercials long   : {c['comm_long']:>10,}  Δ {ch.get('comm_long', 0):>+8,}")
    print(f"  Commercials short  : {c['comm_short']:>10,}  Δ {ch.get('comm_short', 0):>+8,}")
    print(f"  Commercials NET    : {c['comm_net']:>10,}  Δ {ch.get('comm_net', 0):>+8,}")
    print(f"  Large Spec long    : {c['spec_long']:>10,}  Δ {ch.get('nc_long', 0):>+8,}")
    print(f"  Large Spec short   : {c['spec_short']:>10,}  Δ {ch.get('nc_short', 0):>+8,}")
    print(f"  Large Spec NET     : {c['spec_net']:>10,}  Δ {ch.get('spec_net', 0):>+8,}")
    print("━" * 62)
    print(" Live CFTC data loaded successfully")
except Exception as e:
    print(f"⚠️  CFTC live fetch failed: {e}")
    print("   Check connectivity — CFTC updates Fridays ~15:30 ET")
    cot_latest = None


In [ ]:
print(' Setup complete')


In [ ]:
# ── Verify gold commodity code in bulk files ────────────────────────────────
import io, zipfile
BULK_BASE = 'https://www.cftc.gov/files/dea/history/com_disagg_txt_{year}.zip'
url = BULK_BASE.format(year=2024)
try:
    resp = requests.get(url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        fname = [f for f in z.namelist() if f.endswith('.txt') or f.endswith('.csv')][0]
        with z.open(fname) as f:
            df_sample = pd.read_csv(f, low_memory=False)
    gold_like = df_sample[df_sample['Market_and_Exchange_Names'].str.contains('GOLD', case=False, na=False)]
    print("── GOLD-related rows (unique name + code) ──")
    print(gold_like[['Market_and_Exchange_Names','CFTC_Commodity_Code']].drop_duplicates().to_string())
except Exception as e:
    print(f"Code-check skipped: {e}")


In [ ]:
# ── CELL 5: Build rolling COT history + compute Williams COT Index ──────────
# FIX: comm_net / spec_net are now computed from the disaggregated columns
#      (not from the live-page row which had NaN on first append).
#      mm_score is now defined here (was missing → scorecard NameError).

BULK_BASE_FUT = 'https://www.cftc.gov/files/dea/history/fut_disagg_txt_{year}.zip'

def fetch_cot_year(year):
    url = BULK_BASE_FUT.format(year=year)
    resp = requests.get(url, timeout=45, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        fname = [f for f in z.namelist() if f.endswith('.txt') or f.endswith('.csv')][0]
        print(f"  zip({year}) → {fname}")
        with z.open(fname) as f:
            return pd.read_csv(f, low_memory=False)

def find_date_col(df):
    for c in ['As_of_Date_in_Form_YYMMDD','As_of_Date_In_Form_YYMMDD',
              'Report_Date_as_YYYY-MM-DD','Report_Date_as_MM_DD_YYYY']:
        if c in df.columns: return c
    for c in df.columns:
        if 'date' in c.lower(): return c
    raise KeyError(f"No date column found. Cols: {list(df.columns)[:10]}")

def parse_date_col(series, col_name):
    s = series.astype(str).str.strip()
    if 'YYMMDD' in col_name.upper():
        return pd.to_datetime(s, format='%y%m%d', errors='coerce')
    return pd.to_datetime(s, errors='coerce')

def find_col(df, candidates, label):
    for c in candidates:
        if c in df.columns: return c
    avail = [c for c in df.columns if any(k in c for k in ['Prod','Comm','Money','NonComm','Swap','Interest'])]
    raise KeyError(f"{label} not found. Available: {avail[:15]}")

# Download 4 years
cot_frames = []
current_year = datetime.today().year
for yr in range(current_year - 3, current_year + 1):
    try:
        df_yr = fetch_cot_year(yr)
        df_yr['_code'] = df_yr['CFTC_Contract_Market_Code'].astype(str).str.strip().str.zfill(6)
        gold_yr = df_yr[
            (df_yr['_code'] == '088691') &
            (~df_yr['Market_and_Exchange_Names'].str.contains('MICRO', case=False, na=False))
        ].copy()
        if len(gold_yr) > 0:
            cot_frames.append(gold_yr)
            print(f"  ✓ {yr} — {len(gold_yr)} gold rows")
        else:
            print(f"  ✗ {yr} — 0 rows for code 088691")
    except Exception as e:
        print(f"  ✗ {yr} — {e}")

if not cot_frames:
    raise RuntimeError("Could not load any COT files")

cot_hist = pd.concat(cot_frames, ignore_index=True)

date_col = find_date_col(cot_hist)
print(f"\n  Date column : '{date_col}'")
cot_hist['date'] = parse_date_col(cot_hist[date_col], date_col)
cot_hist = cot_hist.dropna(subset=['date']).sort_values('date')

OI_COL = find_col(cot_hist, ['Open_Interest_All','OI_All','Open_Interest'], 'Open Interest')
CL_COL = find_col(cot_hist, ['Prod_Merc_Positions_Long_All','Comm_Positions_Long_All','Comm_Long_All'], 'Comm Long')
CS_COL = find_col(cot_hist, ['Prod_Merc_Positions_Short_All','Comm_Positions_Short_All','Comm_Short_All'], 'Comm Short')
SL_COL = find_col(cot_hist, ['M_Money_Positions_Long_All','NonComm_Positions_Long_All','NonComm_Long_All'], 'Spec Long')
SS_COL = find_col(cot_hist, ['M_Money_Positions_Short_All','NonComm_Positions_Short_All','NonComm_Short_All'], 'Spec Short')
print(f"  OI: {OI_COL}  |  Comm: {CL_COL}/{CS_COL}  |  Spec: {SL_COL}/{SS_COL}")

cot = pd.DataFrame({
    'oi'        : pd.to_numeric(cot_hist[OI_COL],  errors='coerce'),
    'comm_long' : pd.to_numeric(cot_hist[CL_COL],  errors='coerce'),
    'comm_short': pd.to_numeric(cot_hist[CS_COL],  errors='coerce'),
    'spec_long' : pd.to_numeric(cot_hist[SL_COL],  errors='coerce'),
    'spec_short': pd.to_numeric(cot_hist[SS_COL],  errors='coerce'),
}, index=pd.to_datetime(cot_hist['date'].values))
cot.index.name = 'date'
cot = cot.sort_index()

# FIX: compute nets from columns (disaggregated files always have these filled)
cot['comm_net'] = cot['comm_long'] - cot['comm_short']
cot['spec_net']  = cot['spec_long'] - cot['spec_short']

# Append live week only if newer AND we have the raw values from cell 2
if cot_latest is not None and cot_latest.get('report_date') is not None:
    live_date = cot_latest['report_date']
    if len(cot) == 0 or live_date > cot.index[-1]:
        live_row = pd.DataFrame([{
            'oi'        : cot_latest['open_interest'],
            'comm_long' : cot_latest['comm_long'],
            'comm_short': cot_latest['comm_short'],
            'comm_net'  : cot_latest['comm_net'],
            'spec_long' : cot_latest['spec_long'],
            'spec_short': cot_latest['spec_short'],
            'spec_net'  : cot_latest['spec_net'],
        }], index=[pd.Timestamp(live_date)])
        cot = pd.concat([cot, live_row]).sort_index()
        print(f"\n  ✓ Live week ({live_date.date()}) appended — total rows: {len(cot)}")
    else:
        print(f"\n  Live week already in history — total rows: {len(cot)}")

# Williams COT Index (rolling 0-100)
IDEAL_WINDOW = 156
WINDOW = min(IDEAL_WINDOW, len(cot) - 1)
if WINDOW < IDEAL_WINDOW:
    print(f"  ⚠ Only {len(cot)} rows — using {WINDOW}-week window")

def cot_index(series, window):
    mn = series.rolling(window, min_periods=2).min()
    mx = series.rolling(window, min_periods=2).max()
    return ((series - mn) / (mx - mn).replace(0, np.nan) * 100).round(1)

cot['comm_index'] = cot_index(cot['comm_net'], WINDOW)
cot['spec_index']  = cot_index(cot['spec_net'],  WINDOW)
cot['mm_index']   = cot['spec_index']   # alias for chart

latest   = cot.iloc[-1]
comm_idx = float(latest['comm_index']) if pd.notna(latest['comm_index']) else 50.0
spec_idx  = float(latest['spec_index'])  if pd.notna(latest['spec_index'])  else 50.0
comm_net_val = latest['comm_net']
spec_net_val  = latest['spec_net']

suffix_comm = ' ← BEARISH EXTREME' if comm_idx >= 80 else (' ← BULLISH EXTREME' if comm_idx <= 20 else '')
suffix_spec  = ' ← CROWDED LONG'   if spec_idx  >= 80 else (' ← WASHED OUT'     if spec_idx  <= 20 else '')

# FIX: define both comm_score AND mm_score (was missing, causing NameError in scorecard)
comm_score = 1 if comm_idx <= 20 else (-1 if comm_idx >= 80 else 0)
spec_score  = 1 if spec_idx  <= 20 else (-1 if spec_idx  >= 80 else 0)
mm_score   = spec_score   # Managed Money = Large Spec in this framework
cot_signal = comm_score + spec_score

label_map = {2:'🟢 BULLISH', 1:'🟡 Mild bullish', 0:'⬜ Neutral', -1:'🟡 Mild bearish', -2:'🔴 BEARISH'}

print('\n' + '━'*62)
print(f'  COT INDEX SUMMARY  ({WINDOW}-week rolling, {len(cot)} rows)')
print('━'*62)
print(f"  Report date    : {cot.index[-1].strftime('%d %b %Y')}")
cn_str = f"{int(comm_net_val):>+10,}" if pd.notna(comm_net_val) else "         N/A"
sn_str  = f"{int(spec_net_val):>+10,}"  if pd.notna(spec_net_val)  else "         N/A"
print(f"  Commercials net: {cn_str}  Index: {comm_idx:.1f}/100{suffix_comm}")
print(f"  Large Spec net : {sn_str}  Index: {spec_idx:.1f}/100{suffix_spec}")
print(f"  COT signal     : {label_map[cot_signal]}  (comm_score={comm_score:+d}  mm_score={mm_score:+d})")
print('━'*62)


In [ ]:
# ── DIAGNOSTIC: Find actual column names in cot_hist ─────────────────────────
print(f"cot_hist shape: {cot_hist.shape}")
print(f"\nAll columns ({len(cot_hist.columns)}):")
for c in cot_hist.columns:
    print(f"  {c}")


In [ ]:
# ── CELL 5: Build rolling COT history + compute Williams COT Index ──────────

BULK_BASE_FUT = 'https://www.cftc.gov/files/dea/history/fut_disagg_txt_{year}.zip'

def fetch_cot_year(year):
    url = BULK_BASE_FUT.format(year=year)
    resp = requests.get(url, timeout=45, headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        fname = [f for f in z.namelist() if f.endswith('.txt') or f.endswith('.csv')][0]
        print(f"  zip({year}) → {fname}")
        with z.open(fname) as f:
            # ── FIX: CFTC files use comma-thousands separators in numeric cols ──
            return pd.read_csv(f, low_memory=False, thousands=',')

def find_date_col(df):
    for c in ['As_of_Date_In_Form_YYMMDD', 'As_of_Date_in_Form_YYMMDD',
              'Report_Date_as_YYYY-MM-DD', 'Report_Date_as_MM_DD_YYYY']:
        if c in df.columns: return c
    for c in df.columns:
        if 'date' in c.lower(): return c
    raise KeyError(f"No date column found. Cols: {list(df.columns)[:10]}")

def parse_date_col(series, col_name):
    s = series.astype(str).str.strip()
    if 'YYMMDD' in col_name.upper():
        return pd.to_datetime(s, format='%y%m%d', errors='coerce')
    return pd.to_datetime(s, errors='coerce')

# Download 4 years
cot_frames = []
current_year = datetime.today().year
for yr in range(current_year - 3, current_year + 1):
    try:
        df_yr = fetch_cot_year(yr)
        df_yr['_code'] = df_yr['CFTC_Contract_Market_Code'].astype(str).str.strip().str.zfill(6)
        gold_yr = df_yr[
            (df_yr['_code'] == '088691') &
            (~df_yr['Market_and_Exchange_Names'].str.contains('MICRO', case=False, na=False))
        ].copy()
        if len(gold_yr) > 0:
            cot_frames.append(gold_yr)
            print(f"  ✓ {yr} — {len(gold_yr)} gold rows")
            # ── Quick sanity check on numeric cols ──────────────────────────
            sample_val = gold_yr['Prod_Merc_Positions_Long_All'].iloc[0]
            print(f"     sample Prod_Merc_Long value: {repr(sample_val)} (type: {type(sample_val).__name__})")
        else:
            print(f"  ✗ {yr} — 0 rows for code 088691")
    except Exception as e:
        print(f"  ✗ {yr} — {e}")

if not cot_frames:
    raise RuntimeError("Could not load any COT files")

cot_hist = pd.concat(cot_frames, ignore_index=True)

date_col = find_date_col(cot_hist)
print(f"\n  Date column : '{date_col}'")
cot_hist['date'] = parse_date_col(cot_hist[date_col], date_col)
cot_hist = cot_hist.dropna(subset=['date']).sort_values('date')

# ── Hardcode exact column names (confirmed present in your file) ──────────────
CL_COL = 'Prod_Merc_Positions_Long_All'
CS_COL = 'Prod_Merc_Positions_Short_All'
SL_COL = 'M_Money_Positions_Long_All'
SS_COL = 'M_Money_Positions_Short_All'
OI_COL = 'Open_Interest_All'

print(f"  OI: {OI_COL}  |  Comm: {CL_COL}/{CS_COL}  |  Spec: {SL_COL}/{SS_COL}")

# ── Force numeric conversion with explicit cleaning ───────────────────────────
def to_num(series):
    """Strip commas/spaces then convert — handles CFTC formatting edge cases."""
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False).str.strip(),
        errors='coerce'
    )

# ── Build cot DataFrame using positional integer index first, fix later ───────
cot = pd.DataFrame({
    'oi'        : to_num(cot_hist[OI_COL]).values,
    'comm_long' : to_num(cot_hist[CL_COL]).values,
    'comm_short': to_num(cot_hist[CS_COL]).values,
    'spec_long' : to_num(cot_hist[SL_COL]).values,
    'spec_short': to_num(cot_hist[SS_COL]).values,
})

# Compute nets BEFORE setting the DatetimeIndex — avoids any index-alignment NaN
cot['comm_net'] = cot['comm_long'] - cot['comm_short']
cot['spec_net']  = cot['spec_long']  - cot['spec_short']

# Now attach the DatetimeIndex cleanly
dates = pd.to_datetime(
    cot_hist['date'].astype(str).str.strip(),
    errors='coerce'
).values  # use .values to strip any tz info
cot.index = pd.DatetimeIndex(dates).tz_localize(None)  # force tz-naive
cot.index.name = 'date'
cot = cot.sort_index()
cot = cot[~cot.index.duplicated(keep='last')]

# Sanity check
print(f"\n  ✅ cot built: {len(cot)} rows")
print(f"  comm_net valid: {cot['comm_net'].notna().sum()}  |  spec_net valid: {cot['spec_net'].notna().sum()}")
if cot['comm_net'].notna().any():
    print(f"  sample comm_net: {cot['comm_net'].dropna().iloc[-1]:,.0f}")
    print(f"  sample spec_net: {cot['spec_net'].dropna().iloc[-1]:,.0f}")
else:
    print("  ⚠ comm_net still all-NaN — run debug cell to inspect dtypes")

# ── Quick sanity print ────────────────────────────────────────────────────────
print(f"\n  ✅ cot built: {len(cot)} rows")
print(f"  comm_net valid: {cot['comm_net'].notna().sum()}  |  spec_net valid: {cot['spec_net'].notna().sum()}")
print(f"  sample comm_net: {cot['comm_net'].dropna().iloc[-1]:,.0f}")
print(f"  sample spec_net: {cot['spec_net'].dropna().iloc[-1]:,.0f}")

# Append live week only if newer
if cot_latest is not None and cot_latest.get('report_date') is not None:
    live_date = cot_latest['report_date']
    if len(cot) == 0 or live_date > cot.index[-1]:
        live_row = pd.DataFrame([{
            'oi'        : cot_latest['open_interest'],
            'comm_long' : cot_latest['comm_long'],
            'comm_short': cot_latest['comm_short'],
            'comm_net'  : cot_latest['comm_net'],
            'spec_long' : cot_latest['spec_long'],
            'spec_short': cot_latest['spec_short'],
            'spec_net'  : cot_latest['spec_net'],
        }], index=[pd.Timestamp(live_date)])
        cot = pd.concat([cot, live_row]).sort_index()
        cot = cot[~cot.index.duplicated(keep='last')]
        print(f"\n  ✓ Live week ({live_date.date()}) appended — total rows: {len(cot)}")
    else:
        print(f"\n  Live week already in history — total rows: {len(cot)}")

# Williams COT Index (rolling 0-100)
IDEAL_WINDOW = 156
WINDOW = min(IDEAL_WINDOW, len(cot) - 1)
if WINDOW < IDEAL_WINDOW:
    print(f"  ⚠ Only {len(cot)} rows — using {WINDOW}-week window")

def cot_index(series, window):
    mn = series.rolling(window, min_periods=2).min()
    mx = series.rolling(window, min_periods=2).max()
    return ((series - mn) / (mx - mn).replace(0, np.nan) * 100).round(1)

cot['comm_index'] = cot_index(cot['comm_net'], WINDOW)
cot['spec_index']  = cot_index(cot['spec_net'],  WINDOW)
cot['mm_index']   = cot['spec_index']

latest      = cot.iloc[-1]
comm_idx    = float(latest['comm_index']) if pd.notna(latest['comm_index']) else 50.0
spec_idx     = float(latest['spec_index'])  if pd.notna(latest['spec_index'])  else 50.0
comm_net_val = latest['comm_net']
spec_net_val  = latest['spec_net']

suffix_comm = ' ← BEARISH EXTREME' if comm_idx >= 80 else (' ← BULLISH EXTREME' if comm_idx <= 20 else '')
suffix_spec  = ' ← CROWDED LONG'   if spec_idx  >= 80 else (' ← WASHED OUT'     if spec_idx  <= 20 else '')

comm_score = 1 if comm_idx <= 20 else (-1 if comm_idx >= 80 else 0)
spec_score  = 1 if spec_idx  <= 20 else (-1 if spec_idx  >= 80 else 0)
mm_score   = spec_score
cot_signal = comm_score + spec_score

label_map = {2:'🟢 BULLISH', 1:'🟡 Mild bullish', 0:'⬜ Neutral', -1:'🟡 Mild bearish', -2:'🔴 BEARISH'}

print('\n' + '━'*62)
print(f'  COT INDEX SUMMARY  ({WINDOW}-week rolling, {len(cot)} rows)')
print('━'*62)
print(f"  Report date    : {cot.index[-1].strftime('%d %b %Y')}")
cn_str = f"{int(comm_net_val):>+10,}" if pd.notna(comm_net_val) else "         N/A"
sn_str  = f"{int(spec_net_val):>+10,}"  if pd.notna(spec_net_val)  else "         N/A"
print(f"  Commercials net: {cn_str}  Index: {comm_idx:.1f}/100{suffix_comm}")
print(f"  Large Spec net : {sn_str}  Index: {spec_idx:.1f}/100{suffix_spec}")
print(f"  COT signal     : {label_map[cot_signal]}  (comm_score={comm_score:+d}  mm_score={mm_score:+d})")
print('━'*62)


In [ ]:
# ── CELL 6: COT Index chart — Commercials vs Managed Money ──────────────────
# FIX v3: The "insufficient data" message was caused by comm_index / mm_index
#   being all-NaN. This happens when:
#     (a) CFTC bulk downloads fail → few rows → rolling window too large → all NaN
#     (b) comm_net or spec_net is constant over the window → range=0 → 0/0=NaN
#     (c) The live-row append introduced a duplicate index timestamp
#
#   Fixes applied here:
#     1. Deduplicate cot index before plotting (duplicate timestamps → NaN propagation)
#     2. If comm_index / mm_index are all-NaN, recompute in-place with min_periods=1
#        and a shorter fallback window so the chart always renders
#     3. Print a diagnostic summary so you know which fix fired

# ── Diagnostic: how many valid index values do we actually have? ──────────────
print("── COT index diagnostics ──")
for col in ['comm_net', 'spec_net', 'comm_index', 'mm_index']:
    if col in cot.columns:
        n_valid = cot[col].notna().sum()
        print(f"  {col:<14}: {n_valid} valid rows of {len(cot)} total")
    else:
        print(f"  {col:<14}: MISSING COLUMN")

# ── Dedup index (duplicate timestamps cause NaN bleed in rolling) ─────────────
cot_plot = cot[~cot.index.duplicated(keep='last')].copy()
cot_plot = cot_plot.sort_index()
print(f"  rows after dedup: {len(cot_plot)}")

# ── Recompute index columns if they are all-NaN (fallback with shorter window) ─
def cot_index_safe(series, window):
    """Rolling 0-100 index; falls back to shorter window if needed."""
    s = series.dropna()
    if len(s) < 2:
        return pd.Series(np.nan, index=series.index)
    # Try requested window first
    for w in [window, max(2, window // 2), max(2, window // 4), 2]:
        mn = s.rolling(w, min_periods=2).min()
        mx = s.rolling(w, min_periods=2).max()
        rng = (mx - mn).replace(0, np.nan)
        result = ((s - mn) / rng * 100).round(1)
        valid = result.notna().sum()
        if valid >= 2:
            if w < window:
                print(f"  ⚠ Fallback window used: {w} weeks (requested {window})")
            return result.reindex(series.index)
    return pd.Series(np.nan, index=series.index)

for net_col, idx_col in [('comm_net','comm_index'), ('spec_net','mm_index')]:
    if cot_plot[idx_col].notna().sum() < 2:
        print(f"  Recomputing {idx_col} from {net_col} ...")
        cot_plot[idx_col] = cot_index_safe(cot_plot[net_col], WINDOW)
        print(f"  → {cot_plot[idx_col].notna().sum()} valid values after recompute")

# Also keep spec_index in sync
if 'spec_index' in cot_plot.columns:
    cot_plot['spec_index'] = cot_plot['mm_index']

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), facecolor=BG_COLOR)
fig.suptitle('COT Index — Commercials vs Managed Money (3Y rolling)',
             color=GOLD_COLOR, fontsize=13, fontweight='bold')

for ax, col, label, col_color in zip(
        axes,
        ['comm_index', 'mm_index'],
        ['Commercials Index (bearish @ 80+, bullish @ 20-)',
         'Managed Money Index (crowded @ 80+, washed out @ 20-)'],
        [BEAR_COLOR, BULL_COLOR]):

    data = cot_plot[col].dropna()
    data = data[data.index.notna()]

    if len(data) < 2:
        ax.text(0.5, 0.5,
                f'{label}\nInsufficient data — {len(data)} valid points\n'
                f'Check: CFTC bulk download succeeded in Cell 5?',
                transform=ax.transAxes, color='white',
                ha='center', va='center', fontsize=9)
        ax.set_facecolor(BG_COLOR)
        continue

    ax.plot(data.index, data.values, color=col_color, linewidth=1.2, label=label)
    ax.axhline(80, color=BEAR_COLOR, linewidth=0.6, linestyle='--', alpha=0.6, label='Extreme zone (80/20)')
    ax.axhline(20, color=BULL_COLOR, linewidth=0.6, linestyle='--', alpha=0.6)
    ax.fill_between(data.index, data.values, 80, where=(data.values >= 80), alpha=0.25, color=BEAR_COLOR)
    ax.fill_between(data.index, data.values, 20, where=(data.values <= 20), alpha=0.25, color=BULL_COLOR)
    ax.set_ylim(0, 100)
    ax.set_ylabel(label, color='white', fontsize=9)
    ax.set_facecolor(BG_COLOR)
    ax.tick_params(colors='white')
    ax.grid(color=GRID_COLOR, linewidth=0.4)
    ax.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=8)

    last_val = safe_at(data, -1)
    last_idx = data.index[-1]
    ax.axvline(last_idx, color=GOLD_COLOR, linewidth=0.8, linestyle=':')
    ax.annotate(f' {last_val:.0f}', xy=(last_idx, last_val),
                color=GOLD_COLOR, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# ── Large Spec Index Summary  ◄ ADDED (from framework — Tier 1 Positioning) ──
# Extracts and displays the Large Spec (Managed Money) positioning in a
# clear crowding/washout format as described in the Larry Williams framework.

try:
    print("━"*62)
    print("  LARGE SPEC INDEX — Managed Money Crowding Monitor")
    print("━"*62)
    print(f"  Spec (MM) Index   : {spec_idx:.1f}/100")
    print(f"  Spec net position : {int(spec_net_val):>+,} contracts")

    if spec_idx >= 80:
        spec_crowd_label = "🔴 NEAR MAX LONG — crowded, flush risk elevated"
        spec_action      = "Risk of sharp spec liquidation if price rolls — fade rallies"
    elif spec_idx <= 20:
        spec_crowd_label = "🟢 WASHED OUT — positioned for mean-reversion long"
        spec_action      = "Specs capitulated — dip-buy opportunity on confirmation"
    elif spec_idx >= 65:
        spec_crowd_label = "🟡 Elevated — approaching crowded territory"
        spec_action      = "Caution on longs; watch for distribution"
    elif spec_idx <= 35:
        spec_crowd_label = "🟡 Depressed — specs light, fuel for rally if catalyst"
        spec_action      = "Low crowding = cleaner long entry if macro aligns"
    else:
        spec_crowd_label = "⬜ NEUTRAL — no extreme positioning signal"
        spec_action      = "No edge from spec positioning alone"

    print(f"  Crowding signal   : {spec_crowd_label}")
    print(f"  Implication       : {spec_action}")
    print()
    print(f"  Commercials Index : {comm_idx:.1f}/100")
    print(f"  Comm net position : {int(comm_net_val):>+,} contracts")
    if comm_idx >= 80:
        print("  Comm signal       : 🔴 MAX SHORT — commercials most bearish in 3Y (bearish gold)")
    elif comm_idx <= 20:
        print("  Comm signal       : 🟢 MAX LONG (covering) — commercials most bullish in 3Y")
    else:
        print("  Comm signal       : ⬜ Neutral commercial positioning")
    print("━"*62)
    print()
    print("  Williams COT Rule: ≥2 signals aligned (comm extreme + spec extreme)")
    print("  = high-conviction reversal setup. Current: NEUTRAL — no extreme.")

except Exception as e:
    print(f"⚠️  Large Spec Index cell error: {e}")
    print("   Run Cell 5 (COT history) first")


In [ ]:
# ── CELL 7: GLD Options — Put/Call ratio & gamma walls ──────────────────────
gld_ticker = yf.Ticker('GLD')
spot_gld   = float(gld_ticker.history(period='1d')['Close'].iloc[-1])
print(f"GLD spot: ${spot_gld:.2f}")

try:
    expirations = gld_ticker.options
    print(f"Expirations available: {expirations[:8]}")

    all_calls, all_puts = [], []
    for exp in expirations[:3]:
        chain = gld_ticker.option_chain(exp)
        c = chain.calls[['strike','openInterest','impliedVolatility']].copy()
        p = chain.puts [['strike','openInterest','impliedVolatility']].copy()
        c['expiry'] = p['expiry'] = exp
        all_calls.append(c)
        all_puts.append(p)

    calls = pd.concat(all_calls).groupby('strike')[['openInterest']].sum().rename(columns={'openInterest':'call_oi'})
    puts  = pd.concat(all_puts ).groupby('strike')[['openInterest']].sum().rename(columns={'openInterest':'put_oi'})
    oi = calls.join(puts, how='outer').fillna(0)
    oi['total_oi'] = oi['call_oi'] + oi['put_oi']

    atm = oi[(oi.index >= spot_gld * 0.90) & (oi.index <= spot_gld * 1.10)]
    pc_ratio = atm['put_oi'].sum() / atm['call_oi'].sum() if atm['call_oi'].sum() > 0 else np.nan

    top_walls = oi.nlargest(5, 'total_oi')
    pc_label  = '← BEARISH skew (fear)' if pc_ratio > 1.2 else ('← BULLISH skew' if pc_ratio < 0.8 else '← neutral')
    print(f"  Put/Call ratio (ATM ±10%): {pc_ratio:.2f}  {pc_label}")
    print("  Top gamma walls:")
    for strike, row in top_walls.iterrows():
        dist = (strike / spot_gld - 1) * 100
        print(f"    ${strike:.0f}  ({dist:+.1f}%)  call OI: {row['call_oi']:,.0f}  put OI: {row['put_oi']:,.0f}")

except Exception as e:
    print(f"⚠️  Options fetch failed: {e}")
    pc_ratio = np.nan
    oi = pd.DataFrame()


In [ ]:
# ── CELL 8: GLD Options OI chart — gamma wall visualisation ─────────────────
try:
    if oi.empty:
        raise ValueError("No options data")
    fig, ax = plt.subplots(figsize=(14, 4), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)
    fig.suptitle('GLD Options Open Interest — Gamma Walls (nearest 3 expiries)',
                 color=GOLD_COLOR, fontsize=13, fontweight='bold')
    width = (oi.index[1] - oi.index[0]) * 0.4 if len(oi) > 1 else 1
    ax.bar(oi.index - width/2, oi['call_oi'], width=width, color=BULL_COLOR, alpha=0.7, label='Call OI')
    ax.bar(oi.index + width/2, oi['put_oi'],  width=width, color=BEAR_COLOR, alpha=0.7, label='Put OI')
    ax.axvline(spot_gld, color=GOLD_COLOR, linewidth=1.5, linestyle='--', label=f'GLD spot ${spot_gld:.2f}')
    ax.set_xlim(spot_gld * 0.85, spot_gld * 1.15)
    ax.set_xlabel('Strike', color='white')
    ax.set_ylabel('Open Interest', color='white')
    ax.tick_params(colors='white')
    ax.grid(color=GRID_COLOR, linewidth=0.4, axis='y')
    ax.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=9)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Chart skipped: {e}")


In [ ]:
# ── CELL 13c: Gamma Concentration Summary  ◄ ADDED (from framework screenshot) ──
# Gamma walls / pin clusters from the options OI computed in Cell 7.
# Heavy OI strike = price magnet (dealer hedging pins price).
# If price breaks above heavy call OI → dealer forced to buy → acceleration.

try:
    if oi.empty:
        raise ValueError("Options OI data not loaded — run Cell 7 first")

    spot = spot_gld

    # ── Identify top gamma walls within ±15% of spot ──────────────────────────
    atm_window = oi[(oi.index >= spot * 0.85) & (oi.index <= spot * 1.15)].copy()
    atm_window['total_oi'] = atm_window['call_oi'] + atm_window['put_oi']
    atm_window['gamma_weight'] = atm_window['total_oi'] / atm_window['total_oi'].sum()

    top_gamma = atm_window.nlargest(8, 'total_oi')

    # Separate call wall (above spot, dealer short gamma → must buy on rise)
    # and put wall (below spot, dealer short gamma → must sell on drop)
    call_walls = top_gamma[top_gamma.index > spot].head(3)
    put_walls  = top_gamma[top_gamma.index < spot].head(3)
    pin_strike = atm_window['total_oi'].idxmax()

    print("━"*62)
    print("  GAMMA CONCENTRATION — GLD Options (nearest 3 expiries)")
    print("━"*62)
    print(f"  GLD Spot       : ${spot:.2f}")
    print(f"  Max pain / pin : ${pin_strike:.0f}  (highest total OI = price magnet)")
    print(f"  Distance to pin: {(pin_strike/spot - 1)*100:+.1f}%")
    print()
    print("  🔴 CALL WALLS (above spot — break = dealer forced BUY = acceleration):")
    if len(call_walls) == 0:
        print("     None in range")
    for strike, row in call_walls.iterrows():
        dist = (strike/spot - 1)*100
        print(f"     ${strike:.0f}  ({dist:+.1f}%)  call OI {row['call_oi']:,.0f}  put OI {row['put_oi']:,.0f}  total {row['total_oi']:,.0f}")

    print()
    print("  🟢 PUT WALLS (below spot — breach = dealer forced SELL = acceleration):")
    if len(put_walls) == 0:
        print("     None in range")
    for strike, row in put_walls.sort_index(ascending=False).iterrows():
        dist = (strike/spot - 1)*100
        print(f"     ${strike:.0f}  ({dist:+.1f}%)  call OI {row['call_oi']:,.0f}  put OI {row['put_oi']:,.0f}  total {row['total_oi']:,.0f}")

    print()
    # Gamma regime
    pc_above = atm_window[atm_window.index > spot]['put_oi'].sum()
    pc_below = atm_window[atm_window.index < spot]['call_oi'].sum()
    skew_label = ("🔴 Heavy put OI below — dealer will sell into dips (gravity down)" if
                  atm_window[atm_window.index < spot]['put_oi'].sum() >
                  atm_window[atm_window.index > spot]['call_oi'].sum() * 1.3 else
                  "🟢 Heavy call OI above — dealer buy pressure on breakout" if
                  atm_window[atm_window.index > spot]['call_oi'].sum() >
                  atm_window[atm_window.index < spot]['put_oi'].sum() * 1.3 else
                  "⬜ Balanced gamma — price likely pinned near max pain strike")
    print(f"  Gamma regime   : {skew_label}")
    print("━"*62)

except Exception as e:
    print(f"⚠️  Gamma concentration error: {e}")
    print("   Run Cell 7 (GLD Options) first to load OI data")


In [ ]:
# ── GEX (Gamma Exposure) Calculator for Gold — SpotGamma-style ──────────────
# Requires: yfinance, numpy, pandas, matplotlib (all already imported)
# GEX = OI × Gamma × Spot² × Contract_Multiplier × (-1 for puts, +1 for calls)
# Dealer GEX: assumes dealers are SHORT calls and LONG puts (standard assumption)
# Net GEX > 0 = dealers long gamma = price stabilising (pin risk)
# Net GEX < 0 = dealers short gamma = price destabilising (acceleration risk)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# ── Black-Scholes Greeks ──────────────────────────────────────────────────────
def bs_delta_gamma(S, K, T, r, sigma, option_type='call'):
    """Returns (delta, gamma) for a European option."""
    if T <= 0 or sigma <= 0:
        return (0.0, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    if option_type == 'call':
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1
    return delta, gamma

# ── Fetch GLD options chain ───────────────────────────────────────────────────
print("━"*65)
print("  GEX (GAMMA EXPOSURE) — Gold / GLD  |  SpotGamma-style")
print("━"*65)

gld = yf.Ticker('GLD')
spot = float(gld.history(period='1d')['Close'].iloc[-1])
print(f"  GLD Spot     : ${spot:.2f}")
print(f"  Gold Futures : ~${spot*LIVE_RATIO:.1f}  (GLD × 10 proxy)")

# Risk-free rate proxy (use current 3M T-bill)
try:
    irx = yf.Ticker('^IRX')
    r = float(irx.history(period='5d')['Close'].dropna().iloc[-1]) / 100
except:
    r = 0.053  # fallback
print(f"  Risk-free r  : {r:.3f}")

# ── Loop all near-term expirations ────────────────────────────────────────────
from datetime import datetime, date
TODAY = datetime.today()

all_strikes  = []
all_call_gex = []
all_put_gex  = []
all_net_gex  = []
all_call_oi  = []
all_put_oi   = []

expirations = gld.options
# Use expirations within next 60 days (captures front month + next)
near_expiries = [e for e in expirations
                 if (datetime.strptime(e, '%Y-%m-%d') - TODAY).days <= 60]
print(f"  Expirations  : {near_expiries}")
print()

# Collect per-strike GEX across all expiries
strike_gex = {}   # strike → {'call_gex': x, 'put_gex': x, 'call_oi': x, 'put_oi': x}

CONTRACT_MULTIPLIER = 100  # GLD options: 100 shares per contract

for exp in near_expiries:
    T = (datetime.strptime(exp, '%Y-%m-%d') - TODAY).days / 365.0
    if T <= 0:
        continue

    try:
        chain = gld.option_chain(exp)
        calls = chain.calls[['strike', 'openInterest', 'impliedVolatility']].dropna()
        puts  = chain.puts [['strike', 'openInterest', 'impliedVolatility']].dropna()

        # ── Process calls ────────────────────────────────────────────────────
        for _, row in calls.iterrows():
            K     = float(row['strike'])
            oi    = float(row['openInterest'])
            iv    = float(row['impliedVolatility'])
            if oi == 0 or iv == 0 or iv > 5:
                continue
            _, gamma = bs_delta_gamma(spot, K, T, r, iv, 'call')
            # Dealer is SHORT calls → negative call GEX contribution
            # SpotGamma convention: call GEX = +OI × gamma × spot² × multiplier
            gex = oi * gamma * (spot ** 2) * CONTRACT_MULTIPLIER
            if K not in strike_gex:
                strike_gex[K] = {'call_gex': 0, 'put_gex': 0, 'call_oi': 0, 'put_oi': 0}
            strike_gex[K]['call_gex'] += gex
            strike_gex[K]['call_oi']  += oi

        # ── Process puts ─────────────────────────────────────────────────────
        for _, row in puts.iterrows():
            K     = float(row['strike'])
            oi    = float(row['openInterest'])
            iv    = float(row['impliedVolatility'])
            if oi == 0 or iv == 0 or iv > 5:
                continue
            _, gamma = bs_delta_gamma(spot, K, T, r, iv, 'put')
            # Dealer is LONG puts → negative put GEX contribution
            # SpotGamma convention: put GEX = -OI × gamma × spot² × multiplier
            gex = oi * gamma * (spot ** 2) * CONTRACT_MULTIPLIER
            if K not in strike_gex:
                strike_gex[K] = {'call_gex': 0, 'put_gex': 0, 'call_oi': 0, 'put_oi': 0}
            strike_gex[K]['put_gex'] += gex
            strike_gex[K]['put_oi']  += oi

    except Exception as e:
        print(f"  ⚠ Skipped {exp}: {e}")

# ── Build summary DataFrame ───────────────────────────────────────────────────
gex_df = pd.DataFrame(strike_gex).T
gex_df.index.name = 'strike'
gex_df = gex_df.sort_index()

# Net GEX: call_gex (dealer short) - put_gex (dealer long)
# Positive net = dealers long gamma (stabilising / pin)
# Negative net = dealers short gamma (destabilising / trending)
gex_df['net_gex'] = gex_df['call_gex'] - gex_df['put_gex']

# Scale to millions for readability
gex_df['call_gex_m'] = gex_df['call_gex'] / 1e6
gex_df['put_gex_m']  = gex_df['put_gex']  / 1e6
gex_df['net_gex_m']  = gex_df['net_gex']  / 1e6

# Focus on ±15% around spot
lo = spot * 0.85
hi = spot * 1.15
gex_view = gex_df[(gex_df.index >= lo) & (gex_df.index <= hi)].copy()

# ── Key GEX levels ────────────────────────────────────────────────────────────
total_net_gex = gex_df['net_gex'].sum()

# GEX flip point = strike closest to net_gex = 0 crossing (zero-gamma line)
# Find where cumulative net GEX crosses zero
gex_sorted = gex_df.sort_index()
cumulative  = gex_sorted['net_gex'].cumsum()
sign_changes = cumulative[cumulative.shift(1).mul(cumulative) < 0]
if len(sign_changes) > 0:
    gex_flip = float(sign_changes.index[0])
else:
    # fallback: strike with smallest absolute net_gex
    gex_flip = float(gex_df['net_gex'].abs().idxmin())

# Largest positive GEX strike = biggest call wall = resistance
biggest_call_wall = float(gex_view['call_gex'].idxmax()) if len(gex_view) > 0 else spot
# Largest negative GEX strike = biggest put wall = support
biggest_put_wall  = float(gex_view['put_gex'].idxmax())  if len(gex_view) > 0 else spot

# HVL (High Volatility Level) = strike with max absolute net GEX
hvl = float(gex_df['net_gex'].abs().idxmax())

# Top 5 net GEX levels
top_positive = gex_view.nlargest(5, 'net_gex')[['net_gex_m','call_oi','put_oi']]
top_negative = gex_view.nsmallest(5, 'net_gex')[['net_gex_m','call_oi','put_oi']]

print("━"*65)
print(f"  TOTAL NET GEX     : ${total_net_gex/1e6:+.2f}M")
regime = "🟢 LONG GAMMA — dealers stabilising, expect pin/chop" if total_net_gex > 0 \
    else "🔴 SHORT GAMMA — dealers destabilising, expect trending/acceleration"
print(f"  GAMMA REGIME      : {regime}")
print()
print(f"  GEX FLIP POINT    : ${gex_flip:.2f}  (~${gex_flip*LIVE_RATIO:.0f} gold)")
print(f"    → Above flip: dealers long gamma = price pins / dampened moves")
print(f"    → Below flip: dealers short gamma = moves accelerate")
print()
print(f"  KEY CALL WALL     : ${biggest_call_wall:.2f}  (~${biggest_call_wall*LIVE_RATIO:.0f} gold)  ← resistance")
print(f"  KEY PUT WALL      : ${biggest_put_wall:.2f}  (~${biggest_put_wall*LIVE_RATIO:.0f} gold)  ← support")
print(f"  HVL (max abs GEX) : ${hvl:.2f}  (~${hvl*LIVE_RATIO:.0f} gold)")
print()
print("  TOP POSITIVE GEX STRIKES (pinning / call walls):")
print(f"  {'Strike (GLD)':>14} {'Net GEX ($M)':>14} {'Call OI':>10} {'Put OI':>10}  Gold ~")
for strike, row in top_positive.iterrows():
    print(f"  ${strike:>12.1f}  {row['net_gex_m']:>+13.2f}  {int(row['call_oi']):>10,}  {int(row['put_oi']):>10,}  ~${strike*LIVE_RATIO:,.0f}")
print()
print("  TOP NEGATIVE GEX STRIKES (acceleration / put walls):")
print(f"  {'Strike (GLD)':>14} {'Net GEX ($M)':>14} {'Call OI':>10} {'Put OI':>10}  Gold ~")
for strike, row in top_negative.iterrows():
    print(f"  ${strike:>12.1f}  {row['net_gex_m']:>+13.2f}  {int(row['call_oi']):>10,}  {int(row['put_oi']):>10,}  ~${strike*LIVE_RATIO:,.0f}")
print("━"*65)

# ── GEX Chart — SpotGamma style ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), facecolor=BG_COLOR)
fig.suptitle(f'GEX — Gamma Exposure by Strike  |  GLD ${spot:.2f}  (~Gold ${spot*LIVE_RATIO:,.0f})',
             color=GOLD_COLOR, fontsize=14, fontweight='bold')

# ── Top panel: Net GEX bar chart ─────────────────────────────────────────────
ax1.set_facecolor(BG_COLOR)
bar_colors = [BULL_COLOR if v >= 0 else BEAR_COLOR for v in gex_view['net_gex_m']]
bars = ax1.bar(gex_view.index, gex_view['net_gex_m'],
               color=bar_colors, alpha=0.85, width=0.8)
ax1.axvline(spot,     color=GOLD_COLOR, linewidth=2.0, linestyle='--',
            label=f'GLD Spot ${spot:.2f}')
ax1.axvline(gex_flip, color='white',    linewidth=1.2, linestyle=':',
            label=f'GEX Flip ${gex_flip:.0f}  (~${gex_flip*LIVE_RATIO:,.0f})')
ax1.axhline(0, color='#555555', linewidth=0.8)
ax1.set_ylabel('Net GEX ($M)', color='white', fontsize=11)
ax1.set_title('Net GEX per Strike  |  Green = dealers long gamma (pin)  |  Red = dealers short gamma (move)',
              color='white', fontsize=9)
ax1.tick_params(colors='white')
ax1.grid(color=GRID_COLOR, linewidth=0.3, axis='y')
ax1.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=9)

# Annotate top walls
for strike, row in top_positive.head(3).iterrows():
    ax1.annotate(f"${strike:.0f}\n+${row['net_gex_m']:.1f}M",
                 xy=(strike, row['net_gex_m']),
                 xytext=(0, 6), textcoords='offset points',
                 color=BULL_COLOR, fontsize=7, ha='center', fontweight='bold')
for strike, row in top_negative.head(3).iterrows():
    ax1.annotate(f"${strike:.0f}\n${row['net_gex_m']:.1f}M",
                 xy=(strike, row['net_gex_m']),
                 xytext=(0, -14), textcoords='offset points',
                 color=BEAR_COLOR, fontsize=7, ha='center', fontweight='bold')

# ── Bottom panel: Call GEX vs Put GEX side by side ───────────────────────────
ax2.set_facecolor(BG_COLOR)
width = 0.4
strikes = gex_view.index
ax2.bar(strikes - width/2, gex_view['call_gex_m'],
        width=width, color=BULL_COLOR, alpha=0.75, label='Call GEX (resistance)')
ax2.bar(strikes + width/2, -gex_view['put_gex_m'],
        width=width, color=BEAR_COLOR, alpha=0.75, label='Put GEX (support)')
ax2.axvline(spot,     color=GOLD_COLOR, linewidth=2.0, linestyle='--')
ax2.axvline(gex_flip, color='white',    linewidth=1.2, linestyle=':')
ax2.axhline(0, color='#555555', linewidth=0.8)
ax2.set_xlabel('GLD Strike', color='white', fontsize=11)
ax2.set_ylabel('GEX ($M)', color='white', fontsize=11)
ax2.set_title('Call GEX vs Put GEX by Strike',
              color='white', fontsize=9)
ax2.tick_params(colors='white')
ax2.grid(color=GRID_COLOR, linewidth=0.3, axis='y')
ax2.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=9)

plt.tight_layout()
plt.show()

# ── GEX Summary for trading ───────────────────────────────────────────────────
print()
print("━"*65)
print("  GEX TRADING IMPLICATIONS")
print("━"*65)
print(f"  Spot vs Flip  : GLD ${spot:.2f} vs flip ${gex_flip:.2f}  "
      f"→ {'ABOVE flip = pinning zone' if spot > gex_flip else 'BELOW flip = acceleration zone'}")
print(f"  Nearest call wall (resistance) : GLD ${biggest_call_wall:.0f}  (~gold ${biggest_call_wall*LIVE_RATIO:,.0f})")
print(f"  Nearest put wall (support)     : GLD ${biggest_put_wall:.0f}  (~gold ${biggest_put_wall*LIVE_RATIO:,.0f})")
print()
if total_net_gex > 0:
    print("  📌 LONG GAMMA ENVIRONMENT:")
    print(f"     Price tends to be PINNED between put wall and call wall")
    print(f"     Expect mean-reversion, fade moves toward the walls")
    print(f"     Range: GLD ${biggest_put_wall:.0f}–${biggest_call_wall:.0f}  (~gold ${biggest_put_wall*LIVE_RATIO:,.0f}–${biggest_call_wall*LIVE_RATIO:,.0f})")
else:
    print("  ⚡ SHORT GAMMA ENVIRONMENT:")
    print(f"     Price moves tend to ACCELERATE away from the flip point")
    print(f"     Trend-following works better than mean-reversion")
    print(f"     Watch for vol expansion and faster-than-expected moves")
print("━"*65)


Tier 2 · Macro Triggers — Calendar, Real Yields, DXY, CB Demand & Geo Risk
¶
Tickers: 
GC=F
, 
TIP
, 
^TNX
, 
UUP
, 
GLD
, 
^VIX

In [ ]:
# ── CELL 9: Fetch all macro tickers ─────────────────────────────────────────
TICKERS = {
    'GC=F'    : 'Gold futures',
    'TIP'     : 'TIPS ETF (real yield proxy)',
    'UUP'     : 'DXY proxy (ETF)',
    'GLD'     : 'Gold ETF',
    '^VIX'    : 'VIX',
    'ZB=F'    : '30Y Bond futures',
    'EURUSD=X': 'EUR/USD (inverse dollar)',
    '^TNX'    : '10Y Treasury yield',
}
raw = yf.download(list(TICKERS.keys()), start=START_1Y, auto_adjust=True, progress=False)
closes = {}
for t in TICKERS:
    try:
        s = raw['Close'][t].dropna() if isinstance(raw.columns, pd.MultiIndex) else raw['Close'].dropna()
        if len(s) > 0:
            closes[t] = s
            print(f"  ✓ {t:<12} {TICKERS[t]:<32} last: {s.iloc[-1]:.4f}")
        else:
            print(f"  ✗ {t:<12} empty")
    except Exception as ex:
        print(f"  ✗ {t:<12} {ex}")

gold = closes.get('GC=F', closes.get('GLD'))
if gold is None:
    print("⚠️  No gold data — check ticker availability")


In [ ]:
# ── CELL 10: Real yield proxy + macro regime snapshot ───────────────────────
try:
    tip = closes['TIP']
    vix = closes['^VIX']
    uup = closes['UUP']
    gc  = closes['GC=F']

    macro = pd.DataFrame({'gold': gc, 'tip': tip, 'vix': vix, 'uup': uup}).dropna()
    macro['gold_tip_corr'] = macro['gold'].rolling(63).corr(macro['tip'])
    macro['gold_uup_corr'] = macro['gold'].rolling(63).corr(macro['uup'])

    latest_m = macro.iloc[-1]
    wk = macro.iloc[-6:] if len(macro) >= 6 else macro
    gold_wk = (wk['gold'].iloc[-1] / wk['gold'].iloc[0] - 1) * 100
    tip_wk  = (wk['tip'].iloc[-1]  / wk['tip'].iloc[0]  - 1) * 100
    uup_wk  = (wk['uup'].iloc[-1]  / wk['uup'].iloc[0]  - 1) * 100
    vix_lvl = float(latest_m['vix'])

    tip_label = '↑ real yields RISING = headwind' if tip_wk < 0 else '↓ real yields FALLING = tailwind'
    uup_label = '↑ dollar UP = bearish gold'      if uup_wk > 0 else '↓ dollar WEAK = bullish gold'
    vix_label = '↑ fear elevated'                  if vix_lvl > 25 else 'calm'

    print('━'*58)
    print('  MACRO REGIME SNAPSHOT — Real Yields + DXY')
    print('━'*58)
    print(f"  Gold (weekly Δ)   : {gold_wk:+.2f}%")
    print(f"  TIPS (weekly Δ)   : {tip_wk:+.2f}%   {tip_label}")
    print(f"  DXY proxy (wk Δ)  : {uup_wk:+.2f}%   {uup_label}")
    print(f"  VIX level         : {vix_lvl:.1f}   {vix_label}")
    print(f"  Gold/TIP 63d corr : {latest_m['gold_tip_corr']:.2f}")
    print(f"  Gold/UUP 63d corr : {latest_m['gold_uup_corr']:.2f}")

    macro_bull  = int(tip_wk > 0) + int(uup_wk < 0) + int(vix_lvl > 25)
    macro_bear  = int(tip_wk < -0.5) + int(uup_wk > 0.5)
    macro_score = macro_bull - macro_bear
    macro_label = '🟢 Supportive' if macro_score >= 2 else ('🔴 Hostile' if macro_score <= -1 else '⬜ Mixed')
    print(f"\n  Macro score : {macro_score:+d}  ({macro_label})")
    print('━'*58)
except Exception as e:
    print(f'⚠️  Macro block error: {e}')
    macro_score = 0


In [ ]:
# ── CELL 10b: CB Demand + Geo Risk  ◄ PREVIOUSLY MISSING ────────────────────
# Central bank buying flows + geopolitical safe-haven demand
# No free real-time API for CB data; this cell gives:
#   1. WGC demand proxy via GLD + IAU ETF flows
#   2. Geo-risk proxy via TLT (flight-to-safety bond demand) and VIX term structure
#   3. Manual input section for WGC monthly report reading

try:
    # ── ETF flow proxy: GLD + IAU net change as CB/institutional demand signal ──
    iau_data = yf.download('IAU', start=START_6M, auto_adjust=True, progress=False)
    tlt_data = yf.download('TLT', start=START_6M, auto_adjust=True, progress=False)

    if hasattr(iau_data, 'columns') and isinstance(iau_data.columns, pd.MultiIndex):
        iau_close = iau_data['Close']['IAU'].dropna() if 'IAU' in iau_data['Close'].columns else iau_data['Close'].iloc[:,0].dropna()
        tlt_close = tlt_data['Close']['TLT'].dropna() if 'TLT' in tlt_data['Close'].columns else tlt_data['Close'].iloc[:,0].dropna()
    else:
        iau_close = iau_data['Close'].dropna()
        tlt_close = tlt_data['Close'].dropna()

    iau_1m = (iau_close.iloc[-1] / iau_close.iloc[-22] - 1) * 100 if len(iau_close) >= 22 else np.nan
    iau_3m = (iau_close.iloc[-1] / iau_close.iloc[-66] - 1) * 100 if len(iau_close) >= 66 else np.nan
    tlt_1m = (tlt_close.iloc[-1] / tlt_close.iloc[-22] - 1) * 100 if len(tlt_close) >= 22 else np.nan

    # GLD/IAU ratio: if GLD outperforms IAU, larger-lot (institutional/CB) activity likely
    gld_close = closes.get('GLD', iau_close)
    if len(gld_close) >= 22 and len(iau_close) >= 22:
        ratio = (gld_close / iau_close).dropna()
        ratio_chg = (ratio.iloc[-1] / ratio.iloc[-22] - 1) * 100
        ratio_label = 'GLD leading → large-lot demand' if ratio_chg > 0.5 else (
                      'IAU leading → retail demand'    if ratio_chg < -0.5 else 'in line')
    else:
        ratio_chg, ratio_label = 0.0, 'insufficient data'

    # TLT flight-to-safety signal
    tlt_label = '↑ safe-haven bond buying = geo risk ON'  if tlt_1m and tlt_1m > 2 else (
                '↓ bonds selling = risk-on rotation'      if tlt_1m and tlt_1m < -2 else 'neutral')

    print('━'*62)
    print('  CB DEMAND + GEO RISK MONITOR')
    print('━'*62)
    print(f"  IAU (gold ETF) 1M Δ : {iau_1m:+.2f}%" if pd.notna(iau_1m) else "  IAU 1M : N/A")
    print(f"  IAU (gold ETF) 3M Δ : {iau_3m:+.2f}%" if pd.notna(iau_3m) else "  IAU 3M : N/A")
    print(f"  GLD/IAU ratio 1M Δ  : {ratio_chg:+.2f}%  → {ratio_label}")
    print(f"  TLT (bonds) 1M Δ    : {tlt_1m:+.2f}%  → {tlt_label}" if pd.notna(tlt_1m) else "  TLT 1M : N/A")
    print('━'*62)

    # Demand regime score
    cb_geo_bull = int(pd.notna(iau_1m) and iau_1m > 3) + int(pd.notna(tlt_1m) and tlt_1m > 2) + int(ratio_chg > 0.5)
    cb_geo_bear = int(pd.notna(iau_1m) and iau_1m < -3) + int(pd.notna(tlt_1m) and tlt_1m < -2)
    cb_geo_score = cb_geo_bull - cb_geo_bear
    cb_geo_label = '🟢 CB/institutional demand elevated' if cb_geo_score >= 2 else (
                   '🔴 Outflows / risk-on rotation'     if cb_geo_score <= -1 else '⬜ Neutral')
    print(f"  CB+Geo score : {cb_geo_score:+d}  ({cb_geo_label})")

except Exception as e:
    print(f"⚠️  CB demand cell error: {e}")
    cb_geo_score = 0

# ── Manual input section (fill each Sunday from WGC / Reuters / Bloomberg) ──
print("\n  ── Manual CB & Geo inputs (update each Sunday) ──")
CB_MANUAL = {
    'WGC_monthly_tonnes'  : '-30t net (Mar 2026, WGC) — first monthly outflow since 2023; but Q1 2026 total = +244t (+3% YoY); 17 consec months of net purchase streak',
    'top_buyers'          : 'China PBoC (15+ consec months, total >2300t), Poland NBP (102t in 2025, targeting 700t), Kazakhstan, Brazil, Indonesia & Malaysia (returning buyers)',
    'geo_hotspots'        : 'US-Iran conflict ongoing (Feb 28 US+Israel strikes killed Khamenei); Iran retaliatory missile strikes on US Gulf bases (Qatar, UAE, Jordan, Kuwait); ceasefire talks "not there yet" (Rubio May 2026); oil at 4-year highs fueling inflation',
    'sanctions_flow'      : 'Iran sanctions tightened post-conflict; Russia $300B reserves frozen since 2022 remains key CB de-dollarisation catalyst; EM central banks accelerating gold accumulation as reserve diversification',
    'cb_forward_guidance' : 'Fed on hold 3.50-3.75% (97.4% prob no June cut per CME); rate HIKE risk rising — oil-driven inflation, ~55% prob of hike before year-end; Moody\'s US downgrade (May 2025) sustaining de-dollarisation; US GDP Q1 data due May 28',
}
for k, v in CB_MANUAL.items():
    print(f"    {k:<26}: {v if v else '(empty — fill manually)'}")


In [ ]:
# ── CELL 11: Macro 4-panel chart ────────────────────────────────────────────
try:
    fig = plt.figure(figsize=(16, 8), facecolor=BG_COLOR)
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)
    panels = [
        ('gold', GOLD_COLOR,  'Gold futures (GC=F)'),
        ('tip',  '#00BFFF',   'TIPS ETF — real yield proxy'),
        ('uup',  '#FF8C00',   'DXY proxy (UUP)'),
        ('vix',  '#FF4488',   'VIX'),
    ]
    for idx, (col, color, title) in enumerate(panels):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        ax.set_facecolor(BG_COLOR)
        data = macro[col].dropna()
        ax.plot(data.index, data.values, color=color, linewidth=1.2)
        ax.fill_between(data.index, data.values, data.values.min(), alpha=0.12, color=color)
        ax.set_title(title, color=color, fontsize=10)
        ax.tick_params(colors='white', labelsize=8)
        ax.grid(color=GRID_COLOR, linewidth=0.3)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
    fig.suptitle('Macro Regime — 1 Year', color=GOLD_COLOR, fontsize=13, fontweight='bold')
    plt.show()
except Exception as e:
    print(f"Chart skipped: {e}")


In [ ]:
# ── CELL 12: Weekly macro event calendar — Week of 26 May 2026 ───────────────
# Sources: https://www.forexfactory.com/calendar  |  https://www.investing.com/economic-calendar/
weekly_calendar = pd.DataFrame([
    {'day': 'Monday    May 26', 'event': 'US Memorial Day (markets closed)',      'impact': 'low',    'gold_bias': 'thin liquidity — gap risk'},
    {'day': 'Tuesday   May 27', 'event': 'CB Consumer Confidence + Dallas Fed',   'impact': 'medium', 'gold_bias': 'neutral-bearish if strong'},
    {'day': 'Wednesday May 28', 'event': 'US GDP Q1 (2nd estimate) + Fed Beige Bk','impact': 'HIGH',  'gold_bias': 'bearish gold if GDP beats; bullish if miss'},
    {'day': 'Thursday  May 29', 'event': 'US Jobless Claims + PCE Deflator (Apr)','impact': 'HIGH',   'gold_bias': 'PCE is KEY — hot reading = rate-hike fear → bearish'},
    {'day': 'Friday    May 30', 'event': 'Chicago PMI + Pending Home Sales',      'impact': 'medium', 'gold_bias': 'secondary; watch Iran headlines over weekend'},
])
print('━'*72)
print(f"  WEEK OF 26 MAY 2026 — KEY MACRO EVENTS")
print('━'*72)
print(f"  {'DAY':<22} {'EVENT':<38} {'IMPACT':<8} GOLD BIAS")
print('━'*72)
for _, row in weekly_calendar.iterrows():
    impact_icon = '🔴' if row['impact'] == 'HIGH' else ('🟡' if row['impact'] == 'medium' else '⬜')
    print(f"  {row['day']:<22} {row['event']:<38} {impact_icon} {row['impact']:<6} {row['gold_bias']}")
print('━'*72)
print("\n  🔑 WATCH THIS WEEK: PCE Deflator (Thu) + GDP (Wed) + Iran ceasefire headlines")
print("  ⚠️  US Memorial Day Mon = thin Asian/London session Monday — gap risk on open")


Tier 3 · Options & Derivatives — Volatility Regime
¶
GVZ (gold VIX), implied vol surface, 
Vol skew (25Δ Risk-Reversal)

In [ ]:
# ── CELL 13: GVZ (CBOE Gold Volatility Index) ───────────────────────────────
gvz_data = yf.download('^GVZ', start=START_1Y, auto_adjust=True, progress=False)
gvz = None
gvz_pct = 50.0   # safe default so scorecard never fails

if not gvz_data.empty:
    try:
        gvz = gvz_data['Close'].dropna() if 'Close' in gvz_data.columns else gvz_data.iloc[:,0].dropna()
        if hasattr(gvz, 'iloc') and gvz.ndim > 1:
            gvz = gvz.iloc[:, 0]
        gvz_now  = safe_at(gvz, -1)
        gvz_pct  = float(gvz.rank(pct=True).iloc[-1] * 100)
        gvz_52lo = float(gvz.min())
        gvz_52hi = float(gvz.max())
        regime   = ('🔴 COILED — explosive move likely' if gvz_pct < 20 else
                    '🟡 Subdued — below-avg vol'        if gvz_pct < 40 else
                    '⬜ Normal vol environment'           if gvz_pct < 70 else
                    '🟢 Elevated fear — trending')
        print('━'*55)
        print('  GVZ — GOLD VOLATILITY INDEX')
        print('━'*55)
        print(f"  Current GVZ      : {gvz_now:.1f}")
        print(f"  1Y range         : {gvz_52lo:.1f} – {gvz_52hi:.1f}")
        print(f"  Percentile rank  : {gvz_pct:.0f}th")
        print(f"  Regime           : {regime}")
        print('━'*55)
    except Exception as e:
        print(f'⚠️  GVZ parse error: {e}')
else:
    print('⚠️  GVZ not available — using GLD 30d realized vol as proxy')
    if 'GLD' in closes:
        gld_ret = np.log(closes['GLD'] / closes['GLD'].shift(1))
        rvol_30 = gld_ret.rolling(21).std() * np.sqrt(252) * 100
        gvz     = rvol_30.dropna()
        gvz_pct = float(rvol_30.rank(pct=True).iloc[-1] * 100)
        print(f"  GLD 30d realized vol : {rvol_30.iloc[-1]:.1f}%")
        print(f"  Percentile rank (1Y) : {gvz_pct:.0f}th")


In [ ]:
# ── CELL 13b: Vol Skew — 25Δ Risk-Reversal  ◄ PREVIOUSLY MISSING ────────────
# The 25-delta Risk-Reversal measures put IV minus call IV at the same delta.
# Positive RR → put skew heavy = fear of downside (bearish tilt).
# Negative RR → call skew = melt-up hedging / upside demand.
#
# yfinance does NOT provide OTC vol surface data.
# Best free approach: compute skew from GLD listed options (proxy).

try:
    if oi.empty:
        raise ValueError("options OI data not loaded — run Cell 7 first")

    # Re-fetch nearest expiry option chain for IV data
    gld_ticker_skew = yf.Ticker('GLD')
    exps = gld_ticker_skew.options
    if not exps:
        raise ValueError("No GLD options expirations")

    # Use the 2nd expiry (more stable than front week) if available
    exp_to_use = exps[1] if len(exps) > 1 else exps[0]
    chain_skew = gld_ticker_skew.option_chain(exp_to_use)
    calls_sk = chain_skew.calls[['strike','impliedVolatility','openInterest']].dropna()
    puts_sk  = chain_skew.puts [['strike','impliedVolatility','openInterest']].dropna()

    spot = spot_gld

    # Find ~25Δ strikes: approx 5-7% OTM for a 30-day option
    # Simple proxy: call at spot*1.06, put at spot*0.94
    call_target = spot * 1.06
    put_target  = spot * 0.94

    call_25d = calls_sk.iloc[(calls_sk['strike'] - call_target).abs().argsort()[:1]]
    put_25d  = puts_sk.iloc [(puts_sk['strike']  - put_target).abs().argsort()[:1]]

    iv_call_25d = safe_at(call_25d['impliedVolatility'], 0) * 100 if len(call_25d) else np.nan
    iv_put_25d  = safe_at(put_25d ['impliedVolatility'], 0) * 100 if len(put_25d)  else np.nan

    # RR = put IV - call IV at 25Δ
    rr_25d = iv_put_25d - iv_call_25d if (pd.notna(iv_put_25d) and pd.notna(iv_call_25d)) else np.nan

    # ATM IV (strike closest to spot)
    atm_call = calls_sk.iloc[(calls_sk['strike'] - spot).abs().argsort()[:1]]
    iv_atm   = safe_at(atm_call['impliedVolatility'], 0) * 100 if len(atm_call) else np.nan

    rr_label = ('🔴 PUT SKEW HEAVY — fear of drop / downside hedging'  if pd.notna(rr_25d) and rr_25d >  3 else
                '🟢 CALL SKEW — melt-up hedging / upside demand'        if pd.notna(rr_25d) and rr_25d < -3 else
                '⬜ Roughly neutral skew')

    print('━'*62)
    print(f'  VOL SKEW — 25Δ Risk-Reversal  (expiry: {exp_to_use})')
    print('━'*62)
    print(f"  ATM implied vol (GLD proxy) : {iv_atm:.1f}%" if pd.notna(iv_atm) else "  ATM IV: N/A")
    print(f"  25Δ Call IV (~strike {call_25d['strike'].iloc[0]:.0f}) : {iv_call_25d:.1f}%" if pd.notna(iv_call_25d) else "  25Δ Call IV: N/A")
    print(f"  25Δ Put  IV (~strike {put_25d['strike'].iloc[0]:.0f}) : {iv_put_25d:.1f}%" if pd.notna(iv_put_25d) else "  25Δ Put  IV: N/A")
    print(f"  25Δ Risk-Reversal (Put-Call) : {rr_25d:+.2f} vol pts" if pd.notna(rr_25d) else "  RR: N/A")
    print(f"  Interpretation              : {rr_label}")
    print('━'*62)

    # Score for scorecard
    vol_skew_score = (-1 if pd.notna(rr_25d) and rr_25d >  3 else
                       1 if pd.notna(rr_25d) and rr_25d < -3 else 0)

    # Plot skew smile
    merged_sk = calls_sk.rename(columns={'impliedVolatility':'call_iv'}).set_index('strike')[['call_iv']].join(
                puts_sk .rename(columns={'impliedVolatility':'put_iv' }).set_index('strike')[['put_iv']], how='inner')
    merged_sk = merged_sk[(merged_sk.index >= spot * 0.88) & (merged_sk.index <= spot * 1.12)]
    merged_sk *= 100  # to percent

    fig, ax = plt.subplots(figsize=(14, 4), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)
    fig.suptitle(f'GLD Vol Skew — IV Smile  (expiry {exp_to_use})',
                 color=GOLD_COLOR, fontsize=13, fontweight='bold')
    ax.plot(merged_sk.index, merged_sk['call_iv'], color=BULL_COLOR, linewidth=1.3, label='Call IV', marker='o', markersize=3)
    ax.plot(merged_sk.index, merged_sk['put_iv'],  color=BEAR_COLOR, linewidth=1.3, label='Put IV',  marker='o', markersize=3)
    ax.axvline(spot, color=GOLD_COLOR, linewidth=1.2, linestyle='--', label=f'Spot ${spot:.2f}')
    ax.set_xlabel('Strike', color='white')
    ax.set_ylabel('Implied Vol (%)', color='white')
    ax.tick_params(colors='white')
    ax.grid(color=GRID_COLOR, linewidth=0.3)
    ax.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=9)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"⚠️  Vol skew cell error: {e}")
    vol_skew_score = 0


In [ ]:
# ── CELL 14: GVZ chart with regime bands ────────────────────────────────────
try:
    if gvz is None or len(gvz) < 5:
        raise ValueError("No GVZ data")
    fig, ax = plt.subplots(figsize=(14, 4), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)
    fig.suptitle('GVZ — Gold Implied Volatility (1 Year)', color=GOLD_COLOR, fontsize=13, fontweight='bold')
    ax.plot(gvz.index, gvz.values, color='#00BFFF', linewidth=1.2)
    q20 = gvz.quantile(0.20)
    q80 = gvz.quantile(0.80)
    ax.axhline(q20, color=BEAR_COLOR, linewidth=0.8, linestyle='--', label=f'20th pct ({q20:.1f}) — coiled')
    ax.axhline(q80, color=BULL_COLOR, linewidth=0.8, linestyle='--', label=f'80th pct ({q80:.1f}) — fear')
    ax.fill_between(gvz.index, gvz.values, q20, where=(gvz.values <= q20), color=BEAR_COLOR, alpha=0.2)
    ax.fill_between(gvz.index, gvz.values, q80, where=(gvz.values >= q80), color=BULL_COLOR, alpha=0.2)
    ax.set_ylabel('GVZ', color='white')
    ax.tick_params(colors='white')
    ax.grid(color=GRID_COLOR, linewidth=0.3)
    ax.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=8)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Chart skipped: {e}")


Tier 4 · Technical Structure — Weekly Levels, Market Profile & Regime Filter
¶
Prior week H/L/Close · Monthly VWAP · Naked POCs · 
Market Profile (TPO/value area)
 · ADX · EMA stack

In [ ]:
# ── CELL 15: Weekly technical levels + AVWAP ────────────────────────────────
gc_weekly = yf.download('GC=F', start=START_1Y, interval='1wk', auto_adjust=True, progress=False)
if hasattr(gc_weekly, 'columns') and isinstance(gc_weekly.columns, pd.MultiIndex):
    gc_weekly.columns = gc_weekly.columns.get_level_values(0)
gc_weekly = gc_weekly.dropna(subset=['Close'])

pw = gc_weekly.iloc[-2]   # prior completed week
pw_high  = float(pw['High'])
pw_low   = float(pw['Low'])
pw_close = float(pw['Close'])
pw_mid   = (pw_high + pw_low) / 2

# ── Daily data (6M) for VWAP & indicators ────────────────────────────────────
gc_d = yf.download('GC=F', start=START_6M, interval='1d', auto_adjust=True, progress=False)
if hasattr(gc_d, 'columns') and isinstance(gc_d.columns, pd.MultiIndex):
    gc_d.columns = gc_d.columns.get_level_values(0)
gc_d = gc_d.dropna(subset=['Close','Volume'])
gc_d['tp']  = (gc_d['High'] + gc_d['Low'] + gc_d['Close']) / 3
gc_d['tpv'] = gc_d['tp'] * gc_d['Volume']

# AVWAP from 6M start
avwap_6m  = gc_d['tpv'].cumsum() / gc_d['Volume'].cumsum()
avwap_now = safe_at(avwap_6m, -1)
gold_now  = safe_at(gc_d['Close'], -1)

print("━"*55)
print("  WEEKLY TECHNICAL LEVELS")
print("━"*55)
print(f"  Gold spot         : ${gold_now:,.1f}")
print(f"  Prior week high   : ${pw_high:,.1f}  {'← resistance'              if gold_now < pw_high else '← BROKEN → momentum'}")
print(f"  Prior week low    : ${pw_low:,.1f}  {'← support'                 if gold_now > pw_low  else '← BROKEN → weakness'}")
print(f"  Prior week close  : ${pw_close:,.1f}")
print(f"  Prior week mid    : ${pw_mid:,.1f}")
print(f"  AVWAP (6M)        : ${avwap_now:,.1f}  {'← above = bullish' if gold_now > avwap_now else '← below = bearish'}")
print("━"*55)


In [ ]:
# ── CELL 15b: Monthly VWAP + Naked POCs  ◄ PREVIOUSLY MISSING ──────────────
# Monthly VWAP: anchored to the 1st trading day of each of the last 3 months.
# Naked POC: price cluster with highest volume that has never been revisited —
#   computed via a simple volume-at-price histogram on the 6M daily data.

try:
    # ── Monthly AVWAPs ────────────────────────────────────────────────────────
    gc_d2 = gc_d.copy()
    gc_d2['month'] = gc_d2.index.to_period('M')
    recent_months  = sorted(gc_d2['month'].unique())[-3:]

    monthly_vwaps = {}
    for m in recent_months:
        subset = gc_d2[gc_d2['month'] == m]
        if len(subset) < 3:
            continue
        # anchor to first bar of month
        cum_tpv = subset['tpv'].cumsum()
        cum_vol = subset['Volume'].cumsum()
        vwap_m  = cum_tpv / cum_vol
        monthly_vwaps[str(m)] = safe_at(vwap_m, -1)

    print("━"*55)
    print("  MONTHLY VWAP LEVELS")
    print("━"*55)
    for mo, vw in monthly_vwaps.items():
        rel = 'above ↑' if gold_now > vw else 'below ↓'
        print(f"  {mo} VWAP : ${vw:,.1f}  ({rel})")
    print("━"*55)

    # ── Naked POC via Volume-at-Price histogram ───────────────────────────────
    # Bin price into 0.2% buckets; find top-volume buckets not revisited since peak
    price_range  = gc_d2['Close'].max() - gc_d2['Close'].min()
    n_bins       = max(50, int(price_range / (gc_d2['Close'].mean() * 0.002)))
    gc_d2['bin'] = pd.cut(gc_d2['Close'], bins=n_bins)
    vap          = gc_d2.groupby('bin', observed=True)['Volume'].sum().dropna()
    vap.index    = [float(b.mid) for b in vap.index]
    vap          = vap.sort_values(ascending=False)

    # A POC is "naked" if price hasn't crossed it since the last time it was the HVN peak
    # Simple proxy: top-5 volume nodes whose price level wasn't visited in the last 10 sessions
    recent_range = (gc_d2['Low'].iloc[-10:].min(), gc_d2['High'].iloc[-10:].max())
    naked_pocs   = []
    for price_level, vol in vap.head(20).items():
        if not (recent_range[0] <= price_level <= recent_range[1]):
            naked_pocs.append((price_level, vol))
        if len(naked_pocs) >= 3:
            break

    print("\n  NAKED POCs (high-volume nodes not recently visited)")
    print("━"*55)
    if naked_pocs:
        for pl, vl in naked_pocs:
            direction = 'ABOVE — magnet UP' if pl > gold_now else 'BELOW — magnet DOWN'
            print(f"  ${pl:,.1f}  vol cluster: {vl:,.0f}  → {direction}")
    else:
        print("  No clear naked POCs in current 6M range")
    print("━"*55)

    # Plot Volume-at-Price bar chart (horizontal)
    fig, (ax_price, ax_vap) = plt.subplots(1, 2, figsize=(16, 6),
                                            facecolor=BG_COLOR, gridspec_kw={'width_ratios': [3, 1]})
    fig.suptitle('Monthly VWAPs + Volume-at-Price (Naked POC finder)',
                 color=GOLD_COLOR, fontsize=13, fontweight='bold')

    ax_price.set_facecolor(BG_COLOR)
    ax_price.plot(gc_d2.index, gc_d2['Close'], color=GOLD_COLOR, linewidth=1.2, label='GC price')
    colors_mv = ['#00BFFF','#FF8C00','#FF44AA']
    for (mo, vw), col in zip(monthly_vwaps.items(), colors_mv):
        ax_price.axhline(vw, linewidth=1.0, linestyle='--', color=col, alpha=0.85, label=f'{mo} VWAP ${vw:,.0f}')
    for pl, _ in naked_pocs:
        ax_price.axhline(pl, linewidth=0.8, linestyle=':', color='#FFFFFF', alpha=0.5)
        ax_price.annotate(f" Naked POC ${pl:,.0f}", xy=(gc_d2.index[5], pl),
                          color='white', fontsize=7, alpha=0.7)
    ax_price.set_ylabel('Price $', color='white')
    ax_price.tick_params(colors='white')
    ax_price.grid(color=GRID_COLOR, linewidth=0.3)
    ax_price.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=8)

    ax_vap.set_facecolor(BG_COLOR)
    vap_plot = vap.sort_index()
    bar_colors = [BULL_COLOR if p <= gold_now else BEAR_COLOR for p in vap_plot.index]
    ax_vap.barh(vap_plot.index, vap_plot.values, height=price_range/n_bins*0.8,
                color=bar_colors, alpha=0.7)
    ax_vap.axhline(gold_now, color=GOLD_COLOR, linewidth=1.2, linestyle='--', label=f'Spot ${gold_now:,.0f}')
    ax_vap.set_xlabel('Volume', color='white')
    ax_vap.set_ylabel('Price $', color='white')
    ax_vap.set_title('Volume at Price', color='white', fontsize=9)
    ax_vap.tick_params(colors='white', labelsize=7)
    ax_vap.grid(color=GRID_COLOR, linewidth=0.3, axis='x')
    ax_vap.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=7)

    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"⚠️  Monthly VWAP / Naked POC error: {e}")
    import traceback; traceback.print_exc()


In [ ]:
# ── CELL 15c: Market Profile — Balance vs Trend + Value Area Migration  ◄ NEW
# TPO (Time-Price-Opportunity) profile on the last 5 daily sessions.
# Value Area High (VAH), Value Area Low (VAL), Point of Control (POC).
# Migration: if price closes outside the prior week's value area → breakout.

try:
    # Use last 10 sessions for TPO profile
    mp_data = gc_d.iloc[-10:].copy()
    if len(mp_data) < 5:
        raise ValueError("Not enough daily bars for market profile")

    # ── Build TPO count per price bucket ─────────────────────────────────────
    mp_low    = mp_data['Low'].min()
    mp_high   = mp_data['High'].max()
    tick_size = round(mp_data['Close'].mean() * 0.001, 0)   # ~0.1% tick bucket
    tick_size = max(tick_size, 1.0)

    price_levels = np.arange(mp_low, mp_high + tick_size, tick_size)
    tpo_count    = pd.Series(0, index=price_levels)

    for _, bar in mp_data.iterrows():
        touched = tpo_count[(tpo_count.index >= bar['Low']) & (tpo_count.index <= bar['High'])].index
        tpo_count[touched] += 1

    tpo_count = tpo_count[tpo_count > 0].sort_index()

    # POC = price with most TPOs
    poc_price = float(tpo_count.idxmax())

    # Value Area: 70% of total TPO volume centred on POC
    total_tpo   = tpo_count.sum()
    target_tpo  = total_tpo * 0.70
    va_tpos     = tpo_count.sort_values(ascending=False)
    running     = 0
    va_strikes  = []
    for price_lv, cnt in va_tpos.items():
        va_strikes.append(price_lv)
        running += cnt
        if running >= target_tpo:
            break
    vah = max(va_strikes)
    val = min(va_strikes)

    # Market structure
    balanced  = (mp_high - mp_low) < mp_data['Close'].mean() * 0.035  # < 3.5% range = balance
    structure = 'BALANCE (range market — fade extremes)' if balanced else 'TREND (directional — follow breakout)'

    # Value area migration vs prior week
    pw_vah_proxy = pw_high   # simplification: use prior week range as VA proxy
    pw_val_proxy = pw_low
    if gold_now > pw_vah_proxy:
        migration = 'ABOVE prior week VA → bullish migration (buyers in control)'
    elif gold_now < pw_val_proxy:
        migration = 'BELOW prior week VA → bearish migration (sellers in control)'
    else:
        migration = 'INSIDE prior week VA → within balance, await breakout'

    print("━"*60)
    print("  MARKET PROFILE — Last 10 Sessions")
    print("━"*60)
    print(f"  POC (Point of Control) : ${poc_price:,.1f}")
    print(f"  Value Area High (VAH)  : ${vah:,.1f}")
    print(f"  Value Area Low  (VAL)  : ${val:,.1f}")
    print(f"  Value Area width       : ${vah - val:,.1f}  ({(vah-val)/gold_now*100:.1f}%)")
    print(f"  Structure              : {structure}")
    print(f"  VA migration vs PW     : {migration}")
    print(f"  Current price          : ${gold_now:,.1f}  relative to VA: ", end='')
    if   gold_now > vah:  print('ABOVE VAH ← long edge')
    elif gold_now < val:  print('BELOW VAL ← short edge')
    else:                  print('INSIDE VA ← no edge, wait')
    print("━"*60)

    # Score
    mp_score = (1 if gold_now > vah  else
               -1 if gold_now < val  else 0)

    # ── TPO horizontal bar chart ──────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 10), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)
    fig.suptitle('Market Profile — TPO (last 10 sessions)',
                 color=GOLD_COLOR, fontsize=13, fontweight='bold')

    bar_colors = []
    for p in tpo_count.index:
        if p == poc_price:       bar_colors.append(GOLD_COLOR)
        elif val <= p <= vah:    bar_colors.append(BULL_COLOR)
        else:                    bar_colors.append(NEUTRAL_COLOR)

    ax.barh(tpo_count.index, tpo_count.values,
            height=tick_size * 0.85, color=bar_colors, alpha=0.85)
    ax.axhline(gold_now, color='white',   linewidth=1.4, linestyle='--', label=f'Current ${gold_now:,.0f}')
    ax.axhline(poc_price, color=GOLD_COLOR, linewidth=1.0, linestyle=':', label=f'POC ${poc_price:,.0f}')
    ax.axhline(vah,       color=BULL_COLOR, linewidth=0.8, linestyle='--', alpha=0.7, label=f'VAH ${vah:,.0f}')
    ax.axhline(val,       color=BEAR_COLOR, linewidth=0.8, linestyle='--', alpha=0.7, label=f'VAL ${val:,.0f}')
    ax.set_xlabel('TPO Count', color='white')
    ax.set_ylabel('Price $', color='white')
    ax.tick_params(colors='white')
    ax.grid(color=GRID_COLOR, linewidth=0.3, axis='x')
    ax.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=9)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"⚠️  Market Profile error: {e}")
    import traceback; traceback.print_exc()
    mp_score = 0


In [ ]:
# ── CELL 16: ADX + EMA regime filter ────────────────────────────────────────
def compute_adx(df, period=14):
    hi, lo, cl = df['High'], df['Low'], df['Close']
    tr     = pd.concat([hi-lo, (hi-cl.shift()).abs(), (lo-cl.shift()).abs()], axis=1).max(axis=1)
    atr    = tr.ewm(alpha=1/period, adjust=False).mean()
    dm_pos = hi.diff().clip(lower=0)
    dm_neg = (-lo.diff()).clip(lower=0)
    dm_pos = dm_pos.where(dm_pos > dm_neg, 0)
    dm_neg = dm_neg.where(dm_neg > dm_pos, 0)
    di_pos = 100 * dm_pos.ewm(alpha=1/period, adjust=False).mean() / atr
    di_neg = 100 * dm_neg.ewm(alpha=1/period, adjust=False).mean() / atr
    dx     = 100 * (di_pos - di_neg).abs() / (di_pos + di_neg).replace(0, np.nan)
    adx    = dx.ewm(alpha=1/period, adjust=False).mean()
    return adx, di_pos, di_neg

try:
    adx, di_pos, di_neg = compute_adx(gc_d)
    ema20  = gc_d['Close'].ewm(span=20,  adjust=False).mean()
    ema50  = gc_d['Close'].ewm(span=50,  adjust=False).mean()
    ema200 = gc_d['Close'].ewm(span=200, adjust=False).mean()

    adx_now = safe_at(adx, -1)
    price   = safe_at(gc_d['Close'], -1)
    e20, e50, e200 = safe_at(ema20, -1), safe_at(ema50, -1), safe_at(ema200, -1)

    if   price > e20 > e50 > e200: ema_regime = '🟢 Full bull stack — trend continuation'
    elif price < e20 < e50 < e200: ema_regime = '🔴 Full bear stack — distribution'
    elif price > e200:              ema_regime = '🟡 Above 200 EMA — bull but choppy'
    else:                            ema_regime = '🔴 Below 200 EMA — macro bear'

    if   adx_now >= 30: adx_regime = f'🟢 Trending (ADX {adx_now:.0f}) — ride the move'
    elif adx_now >= 20: adx_regime = f'🟡 Developing (ADX {adx_now:.0f}) — cautious directional'
    else:                adx_regime = f'⬜ Choppy (ADX {adx_now:.0f}) — range-trade, fade extremes'

    print("━"*55)
    print("  TECHNICAL REGIME — EMA Stack + ADX")
    print("━"*55)
    print(f"  EMA regime  : {ema_regime}")
    print(f"  ADX regime  : {adx_regime}")
    print(f"  EMA20: ${e20:,.0f}  EMA50: ${e50:,.0f}  EMA200: ${e200:,.0f}")
    print("━"*55)
except Exception as e:
    print(f"⚠️  ADX/EMA error: {e}")


In [ ]:
# ── CELL 17: Price + EMA stack + ADX chart ──────────────────────────────────
try:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), facecolor=BG_COLOR,
                                    gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle('Gold — EMA Stack + ADX (6M daily)', color=GOLD_COLOR, fontsize=13, fontweight='bold')
    for ax in (ax1, ax2): ax.set_facecolor(BG_COLOR)

    ax1.plot(gc_d.index, gc_d['Close'], color=GOLD_COLOR, linewidth=1.2, label='GC price')
    ax1.plot(gc_d.index, ema20,  color='#00FF88', linewidth=0.9, linestyle='--', label='EMA20',  alpha=0.8)
    ax1.plot(gc_d.index, ema50,  color='#00BFFF', linewidth=0.9, linestyle='--', label='EMA50',  alpha=0.8)
    ax1.plot(gc_d.index, ema200, color='#FF8C00', linewidth=1.0, linestyle='--', label='EMA200', alpha=0.8)
    ax1.plot(gc_d.index, avwap_6m, color='#FF44AA', linewidth=0.9, linestyle=':', label='AVWAP 6M', alpha=0.8)
    ax1.axhline(pw_high,  color='#FF4444', linewidth=0.7, alpha=0.6, label=f'PW High ${pw_high:,.0f}')
    ax1.axhline(pw_low,   color='#44FF88', linewidth=0.7, alpha=0.6, label=f'PW Low ${pw_low:,.0f}')
    ax1.axhline(pw_close, color='#AAAAAA', linewidth=0.5, linestyle=':', alpha=0.6)
    ax1.set_ylabel('Price $', color='white')
    ax1.tick_params(colors='white')
    ax1.grid(color=GRID_COLOR, linewidth=0.3)
    ax1.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=8, ncol=4)

    ax2.plot(adx.index, adx.values, color='#FF8C00', linewidth=1.1, label='ADX')
    ax2.axhline(30, color=BULL_COLOR,    linewidth=0.6, linestyle='--', alpha=0.7, label='Trend (30)')
    ax2.axhline(20, color=NEUTRAL_COLOR, linewidth=0.6, linestyle='--', alpha=0.5, label='Chop (20)')
    ax2.fill_between(adx.index, adx.values, 30, where=(adx.values >= 30), color=BULL_COLOR, alpha=0.15)
    ax2.set_ylabel('ADX', color='white')
    ax2.set_ylim(0, 60)
    ax2.tick_params(colors='white')
    ax2.grid(color=GRID_COLOR, linewidth=0.3)
    ax2.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=8)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Chart skipped: {e}")


Wild Cards · BTC, Oil & 
SGE Premium (China Demand)
¶
Cross-asset correlations and inflation expectations

In [ ]:
# ── CELL 18: Wild card signals — BTC, Oil ───────────────────────────────────
wc_tickers = {'BTC-USD': 'Bitcoin', 'CL=F': 'WTI Crude', 'GC=F': 'Gold'}
wc_raw = yf.download(list(wc_tickers.keys()), start=START_6M, auto_adjust=True, progress=False)
wc = {}
for t in wc_tickers:
    try:
        s = wc_raw['Close'][t].dropna() if isinstance(wc_raw.columns, pd.MultiIndex) else wc_raw['Close'].dropna()
        if len(s) > 0: wc[t] = s
    except Exception: pass

print("━"*55)
print("  WILD CARD SIGNALS")
print("━"*55)
for t, name in wc_tickers.items():
    if t not in wc or len(wc[t]) < 6: continue
    s = wc[t]
    wk_chg = (s.iloc[-1] / s.iloc[-6]  - 1) * 100 if len(s) >= 6  else np.nan
    mo_chg = (s.iloc[-1] / s.iloc[-22] - 1) * 100 if len(s) >= 22 else np.nan
    print(f"  {name:<12} last: {s.iloc[-1]:>10,.1f}  1W: {wk_chg:+.1f}%  1M: {mo_chg:+.1f}%")

if all(t in wc for t in ['GC=F','BTC-USD','CL=F']):
    corr_df = pd.DataFrame({'gold': wc['GC=F'], 'btc': wc['BTC-USD'], 'oil': wc['CL=F']}).dropna()
    gold_btc_corr = float(corr_df['gold'].rolling(42).corr(corr_df['btc']).iloc[-1])
    gold_oil_corr = float(corr_df['gold'].rolling(42).corr(corr_df['oil']).iloc[-1])
    btc_label = 'risk-on linked' if gold_btc_corr > 0.4 else ('decoupled' if gold_btc_corr < 0 else 'weakly linked')
    oil_label = 'inflation narrative active' if gold_oil_corr > 0.4 else 'decoupled'
    print(f"  Gold/BTC 42d corr : {gold_btc_corr:+.2f}  {btc_label}")
    print(f"  Gold/Oil 42d corr : {gold_oil_corr:+.2f}  {oil_label}")
print("━"*55)


In [ ]:
# ── CELL 18b: SGE Premium (China physical demand)  ◄ PREVIOUSLY MISSING ─────
# Shanghai Gold Exchange (SGE) premium = SGE price - LBMA spot price in USD.
# Premium > $10/oz  → strong Chinese physical demand (bullish directional signal)
# Premium < $0/oz   → Chinese export flow risk (bearish)
# Premium $0-$10    → normal arb band, neutral
#
# No free live API for SGE prices. This cell provides:
#   1. Manual input section (fill from Reuters/Bloomberg/Kitco each Sunday)
#   2. Automated proxy: CNY/USD + Shanghai gold ETF (518880.SS) vs GLD ratio
#   3. Historical context chart using the proxy

try:
    # ── SGE proxy: Shanghai Gold ETF (518880.SS) converted to USD ────────────
    sge_proxy_raw = yf.download(['518880.SS', 'CNYUSD=X'], start=START_6M,
                                  auto_adjust=True, progress=False)

    if hasattr(sge_proxy_raw, 'columns') and isinstance(sge_proxy_raw.columns, pd.MultiIndex):
        sge_etf_cny = sge_proxy_raw['Close']['518880.SS'].dropna()
        cny_usd     = sge_proxy_raw['Close']['CNYUSD=X'].dropna()
    else:
        raise ValueError("Multi-index not available for SGE proxy")

    # Align and convert
    sge_df = pd.DataFrame({'sge_cny': sge_etf_cny, 'cny_usd': cny_usd}).dropna()
    # 518880 is in CNY per gram → convert to USD per troy oz
    GRAMS_PER_OZ = 31.1035
    sge_df['sge_usd_oz'] = sge_df['sge_cny'] * sge_df['cny_usd'] * GRAMS_PER_OZ

    # GLD proxy (tracks LBMA spot, 1/10 oz per share → *10)
    gld_usd = closes.get('GLD', pd.Series(dtype=float))
    if len(gld_usd) > 0:
        sge_merged = sge_df.join(gld_usd.rename('gld_usd_10th'), how='inner')
        sge_merged['lbma_proxy_oz'] = sge_merged['gld_usd_10th'] * LIVE_RATIO   # was * 10
        sge_merged['premium_usd']   = sge_merged['sge_usd_oz'] - sge_merged['lbma_proxy_oz']
        sge_prem_now = safe_at(sge_merged['premium_usd'], -1)
        sge_prem_4w  = float(sge_merged['premium_usd'].rolling(20).mean().iloc[-1])
    else:
        raise ValueError("GLD data unavailable for premium calc")

    prem_label = ('🟢 STRONG PHYSICAL DEMAND — bullish signal'  if sge_prem_now > 10 else
                  '🔴 DISCOUNT — export flow risk, bearish'      if sge_prem_now <  0 else
                  '⬜ Normal arb band — neutral')

    print("━"*62)
    print("  SGE PREMIUM — China Physical Gold Demand")
    print("━"*62)
    print(f"  SGE price (USD/oz proxy) : ${sge_merged['sge_usd_oz'].iloc[-1]:,.1f}")
    print(f"  LBMA proxy (GLD×10)     : ${sge_merged['lbma_proxy_oz'].iloc[-1]:,.1f}")
    print(f"  SGE premium (current)   : ${sge_prem_now:+.1f}/oz")
    print(f"  SGE premium (4W avg)    : ${sge_prem_4w:+.1f}/oz")
    print(f"  Signal                  : {prem_label}")
    print("━"*62)

    sge_score = (1 if sge_prem_now > 10 else (-1 if sge_prem_now < 0 else 0))

    # Chart
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), facecolor=BG_COLOR,
                                    gridspec_kw={'height_ratios':[2,1]})
    fig.suptitle('SGE Premium — China Physical Gold Demand Proxy',
                 color=GOLD_COLOR, fontsize=13, fontweight='bold')
    for ax in (ax1, ax2): ax.set_facecolor(BG_COLOR)

    ax1.plot(sge_merged.index, sge_merged['sge_usd_oz'],       color='#FF8C00',  linewidth=1.1, label='SGE (USD/oz proxy)')
    ax1.plot(sge_merged.index, sge_merged['lbma_proxy_oz'],    color=GOLD_COLOR, linewidth=1.1, label='LBMA proxy (GLD×10)')
    ax1.set_ylabel('Price USD/oz', color='white')
    ax1.tick_params(colors='white')
    ax1.grid(color=GRID_COLOR, linewidth=0.3)
    ax1.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=8)

    prem = sge_merged['premium_usd']
    ax2.bar(prem.index, prem.values,
            color=[BULL_COLOR if v > 0 else BEAR_COLOR for v in prem.values],
            alpha=0.7, width=1, label='SGE premium (USD/oz)')
    ax2.axhline(10, color=BULL_COLOR, linewidth=0.7, linestyle='--', alpha=0.6, label='Demand zone (+$10)')
    ax2.axhline( 0, color='white',    linewidth=0.5, linestyle=':')
    ax2.set_ylabel('Premium USD/oz', color='white')
    ax2.tick_params(colors='white')
    ax2.grid(color=GRID_COLOR, linewidth=0.3)
    ax2.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=8)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"⚠️  SGE proxy error: {e}")
    print("   → Using manual input fallback below")
    sge_score = 0

# ── Manual override — fill each Sunday from Kitco / Reuters ──────────────────
print("\n  ── SGE Manual Input (update each Sunday) ──")
SGE_MANUAL = {
    'sge_premium_usd_oz' : 'CHECK: metalcharts.org/shanghai/xau (proxy calc unreliable this week due to 518880.SS ETF data gap in yfinance; Au(T+D) vs LBMA spot historically near 0 to +15 in normal demand; current environment may show discount on profit-taking)',
    'source_date'        : 'Friday 23 May 2026',
    'cn_demand_narrative': 'Q1 2026 China physical bar demand at all-time quarterly record (WGC 474t globally, 2nd highest ever). PBoC maintained 15+ consec months buying >2300t total. Jewellery tonnage pressured by high prices but investment bars surging. Wedding season (May-Jun) adds physical floor',
    'hk_import_data'     : 'WGC Q1 2026: Asian physical demand record — Western ETF holders sold paper gold while Asian buyers absorbed physical at record prices. Check gold.org/goldhub/data for April monthly update (due ~May 12-14, may already be out)',
}
for k, v in SGE_MANUAL.items():
    print(f"    {k:<26}: {v if v else '(empty — fill manually)'}")
print("\n  SGE links:")
print("    → https://www.sge.com.cn  (Chinese)")
print("    → https://www.kitco.com/gold/world/  (English proxy)")
print("    → WGC monthly demand: https://www.gold.org/goldhub/data")


In [ ]:
# ── CELL 19: Cross-asset normalised chart (rebased to 100) ──────────────────
try:
    base    = corr_df.iloc[0]
    rebased = corr_df / base * 100
    fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG_COLOR)
    ax.set_facecolor(BG_COLOR)
    fig.suptitle('Cross-asset — Gold, BTC, Oil (rebased to 100, 6M)',
                 color=GOLD_COLOR, fontsize=13, fontweight='bold')
    ax.plot(rebased.index, rebased['gold'], color=GOLD_COLOR, linewidth=1.4, label='Gold')
    ax.plot(rebased.index, rebased['btc'],  color='#FF8C00',  linewidth=1.0, label='BTC',     alpha=0.8)
    ax.plot(rebased.index, rebased['oil'],  color='#888888',  linewidth=1.0, label='WTI Oil', alpha=0.8)
    ax.axhline(100, color='#333333', linewidth=0.5, linestyle='--')
    ax.set_ylabel('Rebased (start = 100)', color='white')
    ax.tick_params(colors='white')
    ax.grid(color=GRID_COLOR, linewidth=0.3)
    ax.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Chart skipped: {e}")


Master Scorecard — Williams 4-Signal Threshold
¶
Aggregate all tier scores. 
≥ +4 = valid setup. < 4 = no trade, observe only.

In [ ]:
# ── CELL 20: Master weekly scorecard ────────────────────────────────────────
# FIX: mm_score was undefined (NameError) — now defined in Cell 5.
# FIX: gvz_pct had no safe default — now initialised to 50.0 in Cell 13.
# NEW: vol_skew_score, cb_geo_score, mp_score, sge_score added.

scores = {}

# ── Tier 1: Positioning ───────────────────────────────────────────────────────
try:
    scores['COT Commercials']  = comm_score
    scores['COT Managed Money']= mm_score
    scores['GLD Put/Call']     = -1 if pc_ratio > 1.25 else (1 if pc_ratio < 0.75 else 0)
except Exception:
    scores.setdefault('COT Commercials',   0)
    scores.setdefault('COT Managed Money', 0)
    scores.setdefault('GLD Put/Call',      0)

# ── Tier 2: Macro ─────────────────────────────────────────────────────────────
try:
    scores['TIPS direction'] = 1 if tip_wk > 0.3 else (-1 if tip_wk < -0.3 else 0)
    scores['DXY direction']  = -1 if uup_wk > 0.3 else (1 if uup_wk < -0.3 else 0)
    scores['VIX regime']     = 1 if vix_lvl > 25 else 0
except Exception:
    scores.setdefault('TIPS direction', 0)
    scores.setdefault('DXY direction',  0)
    scores.setdefault('VIX regime',     0)

try:
    scores['CB + Geo demand'] = cb_geo_score
except Exception:
    scores['CB + Geo demand'] = 0

# ── Tier 3: Options & Vol ─────────────────────────────────────────────────────
try:
    scores['GVZ regime']   = -1 if gvz_pct < 20 else (1 if gvz_pct > 70 else 0)
except Exception:
    scores['GVZ regime']   = 0

try:
    scores['Vol Skew 25Δ RR'] = vol_skew_score
except Exception:
    scores['Vol Skew 25Δ RR'] = 0

# ── Tier 4: Technicals ───────────────────────────────────────────────────────
try:
    scores['EMA stack']    = (1 if price > e20 > e50 > e200 else (-1 if price < e20 < e50 < e200 else 0))
    scores['ADX']          = (1 if adx_now >= 30 else (0 if adx_now >= 20 else -1))
    scores['vs AVWAP']     = (1 if gold_now > avwap_now else -1)
    scores['vs PW levels'] = (1 if gold_now > pw_high else (-1 if gold_now < pw_low else 0))
except Exception:
    for k in ['EMA stack','ADX','vs AVWAP','vs PW levels']:
        scores.setdefault(k, 0)

try:
    scores['Market Profile'] = mp_score
except Exception:
    scores['Market Profile'] = 0

# ── Wild cards ────────────────────────────────────────────────────────────────
try:
    scores['Gold/Oil corr'] = (1 if gold_oil_corr > 0.5 and
                               (wc['CL=F'].iloc[-1]/wc['CL=F'].iloc[-6] - 1) > 0.01 else 0)
except Exception:
    scores['Gold/Oil corr'] = 0

try:
    scores['SGE premium'] = sge_score
except Exception:
    scores['SGE premium'] = 0

# ── Print scorecard ───────────────────────────────────────────────────────────
total_bull = sum(v for v in scores.values() if v > 0)
total_bear = sum(v for v in scores.values() if v < 0)
net_score  = sum(scores.values())

print("=" * 62)
print("  GOLD WEEKLY MASTER SCORECARD")
print(f"  {TODAY.strftime('%A %d %B %Y')}")
print("=" * 62)
print(f"  {'SIGNAL':<28} {'SCORE':>6}  BAR")
print("-" * 62)
for name, score in scores.items():
    bar        = '█' * abs(int(score * 2)) if score != 0 else '·'
    color_char = '🟢' if score > 0 else ('🔴' if score < 0 else '⬜')
    print(f"  {color_char} {name:<26} {score:>+5.0f}  {bar}")
print("-" * 62)
print(f"  {'Bullish signals':<28} {total_bull:>+5.0f}")
print(f"  {'Bearish signals':<28} {total_bear:>+5.0f}")
print(f"  {'NET SCORE':<28} {net_score:>+5.0f}")
print("=" * 62)

THRESHOLD = 4
if   net_score >=  THRESHOLD: verdict = f"🟢 VALID BULLISH SETUP  (score {net_score:+.0f} ≥ +{THRESHOLD})"
elif net_score <= -THRESHOLD: verdict = f"🔴 VALID BEARISH SETUP  (score {net_score:+.0f} ≤ -{THRESHOLD})"
elif abs(net_score) >= 2:     verdict = f"🟡 DEVELOPING — watch for confirmation  (score {net_score:+.0f})"
else:                          verdict = f"⬜ NO SETUP — observe only  (score {net_score:+.0f})"

print(f"\n  VERDICT: {verdict}")
print("=" * 62)


In [ ]:
# ── CELL 21: Scorecard visual bar chart ─────────────────────────────────────
# FIX: xlim was hardcoded to (-3, 3) — now dynamic so all new signals fit.
fig, ax = plt.subplots(figsize=(13, max(6, len(scores) * 0.55)), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)
fig.suptitle(f'Gold Weekly Scorecard — Net {net_score:+.0f} | {TODAY.strftime("%d %b %Y")}',
             color=GOLD_COLOR, fontsize=13, fontweight='bold')

labels = list(scores.keys())
values = list(scores.values())
colors = [BULL_COLOR if v > 0 else (BEAR_COLOR if v < 0 else NEUTRAL_COLOR) for v in values]
bars   = ax.barh(labels, values, color=colors, alpha=0.85, height=0.6)

ax.axvline(0, color='white', linewidth=0.8)
ax.axvline( THRESHOLD, color=BULL_COLOR, linewidth=1, linestyle='--', alpha=0.6, label=f'+{THRESHOLD} threshold')
ax.axvline(-THRESHOLD, color=BEAR_COLOR, linewidth=1, linestyle='--', alpha=0.6, label=f'-{THRESHOLD} threshold')

# FIX: dynamic x-axis limit based on actual max score
max_val = max(abs(v) for v in values) if values else 2
ax.set_xlim(-(max(max_val, THRESHOLD) + 0.8), (max(max_val, THRESHOLD) + 0.8))

ax.tick_params(colors='white')
ax.set_xlabel('Score', color='white')
ax.grid(color=GRID_COLOR, linewidth=0.3, axis='x')
ax.legend(facecolor='#111', edgecolor=GRID_COLOR, labelcolor='white', fontsize=9)

for bar, val in zip(bars, values):
    if val != 0:
        ax.text(val + (0.06 if val > 0 else -0.06),
                bar.get_y() + bar.get_height() / 2,
                f'{val:+.0f}', color='white', va='center',
                ha='left' if val > 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()
print(f'\n  ► VERDICT: {verdict}')


In [ ]:
# ── CELL 22: Weekly notes — pre-filled from computed values  ◄ FIXED ─────────
# FIX: was a static empty template; now auto-fills everything computed above.

def safe_str(val, fmt='.2f', fallback='N/A'):
    try:    return format(float(val), fmt)
    except: return fallback

# Auto-populate from computed variables
weekly_notes = {
    'date'              : TODAY.strftime('%Y-%m-%d'),
    'gold_level_close'  : safe_str(gold_now, ',.1f'),
    'prior_week_range'  : f"{safe_str(pw_low,',.1f')} – {safe_str(pw_high,',.1f')}",
    'avwap_6m'          : safe_str(avwap_now, ',.1f'),
    'net_score'         : f"{net_score:+.0f}",
    'verdict'           : verdict,
    'cot_comm_index'    : safe_str(comm_idx, '.1f'),
    'cot_mm_index'      : safe_str(spec_idx, '.1f'),
    'gvz_pct_rank'      : safe_str(gvz_pct, '.0f') + 'th',
    'gvz_regime'        : ('COILED' if gvz_pct < 20 else 'SUBDUED' if gvz_pct < 40 else 'NORMAL' if gvz_pct < 70 else 'ELEVATED'),
    'pc_ratio'          : safe_str(pc_ratio, '.2f'),
    'tip_wk_pct'        : safe_str(tip_wk, '+.2f') + '%',
    'uup_wk_pct'        : safe_str(uup_wk, '+.2f') + '%',
    'vix_level'         : safe_str(vix_lvl, '.1f'),
    'ema_regime'        : ema_regime if 'ema_regime' in dir() else 'N/A',
    'adx_value'         : safe_str(adx_now, '.0f'),
    'mp_poc'            : safe_str(poc_price, ',.1f') if 'poc_price' in dir() else 'N/A',
    'mp_vah'            : safe_str(vah,       ',.1f') if 'vah'       in dir() else 'N/A',
    'mp_val'            : safe_str(val,       ',.1f') if 'val'       in dir() else 'N/A',
    'sge_premium_proxy' : safe_str(sge_prem_now, '+.1f') + ' USD/oz' if 'sge_prem_now' in dir() else 'manual',
    # ── Fill these manually ──────────────────────────────────────────────────
    'key_levels_watch'  : 'S1: 4493-4540 (yearly LWC + Apr-low + 61.8% retrace — KEY FLOOR). S2: 4400-4460 (200d MA zone). S3: 4200-4300 (Fib+psych). R1: 4686 (bearish island reversal flip level). R2: 4769-4784 (50d+100d SMA confluence). R3: 4894 (record HWC). R4: 5000-5025 (major psych + Oct trendline)',
    'cot_narrative'     : 'COT indices near neutral 50 — no extremes in commercials or managed money. Neither a bullish flush nor a crowded-long washout. Spec net has been declining from highs (consistent with recent price weakness). Watch if commercials begin covering shorts aggressively — that is the early bullish reversal tell. Current: no COT edge, follow price action',
    'macro_theme'       : 'Dual headwind: (1) Oil-driven inflation revival (Iran conflict, WTI at 4Y highs) raising Fed rate-hike probability to ~55% by year-end; (2) Dollar resilience offsetting safe-haven bid. Offset: Moody\'s US downgrade (May 2025) + EM CB de-dollarisation = structural gold bid. Net macro: HOSTILE near-term — real yield/dollar combo outweighing safe-haven. Gold below AVWAP 4628 = bears in control of macro positioning',
    'geo_wildcard'      : 'US-Iran: Rubio says "slight progress" in Oman-mediated talks — but "not there yet" on peace deal. Binary risk: DEAL → risk-on flush to 4300-4400; ESCALATION → spike to 4750+. Oil at 4Y highs = secondary inflation wildcard amplifying Fed hawkishness. Monitor Sunday/Monday headlines before positioning',
    'bias'              : 'BEAR-RANGE — valid bearish setup (score -4). Price below 200 EMA (macro bear) + below AVWAP 4628. ADX 20 = low trend = no strong directional conviction = range-trade preferred. FADE rallies. No longs while below 4628 AVWAP. Bearish conviction only on break below 4493-4540 support floor',
    'trade_plan'        : 'PRIMARY: Short rallies into 4680-4720 resistance zone (R1 area), stop above 4800 (above 50d SMA), target 4493-4540 S1. If 4493 breaks cleanly on volume → add short, target 4300. SECONDARY: Range fade — long bounce at 4493-4540 (tight stop 4460), target 4680. DO NOT trade Monday (Memorial Day thin). Wait for PCE data Thu before adding size. INVALIDATION: Daily close above 4894',
    'notes'             : 'Week of 25 May 2026. Gold at ~4509-4523 (2nd consec weekly decline, tradingeconomics.com). SGE proxy cell (18b) shows large error — 518880.SS ETF not properly proxying SGE; fill SGE_MANUAL manually from metalcharts.org/shanghai/xau each Sunday. WGC March data: CBs net -30t (first outflow print) but may reverse in April. Q1 overall strong at 244t. Cell 6 COT chart may show insufficient data if CFTC bulk download fails — retry or check CFTC site connectivity. Cell 1 CB score -1 is correct given IAU outflows.',
}

print("="*65)
print(f"  📝  GOLD WEEKLY NOTES  — {TODAY.strftime('%A %d %B %Y')}")
print("="*65)
for k, v in weekly_notes.items():
    filled = v if v else '(empty — fill manually)'
    marker = '✅' if v else '⚠️ '
    print(f"  {marker} {k:<26}: {filled}")
print("="*65)
print("\n💾  Tip: save this output to a markdown/csv log for weekly journal.")
